## ドライブのマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN2')

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import login
login(HF_TOKEN) # ここでトークンを入力してね（Write権限があるトークンが必要だよ）

# 環境設定
  - 標準コード1と同様、まずは環境設定を行います。

In [ ]:
# ============================================================
# 0) 依存関係の固定（Colabの“環境ブレ”対策）
# ============================================================
# Colab（無料版）は、ある日突然プリインストール版が変わり、
# それまで動いていた学習コードが壊れることが頻繁にあります。
# そのため、このセルでは「一度全部消す → 互換が確認できたバージョンを入れ直す」
# という“強制的な再現性確保”をしています。
#
# やりがちな事故：
# - 既に入っているパッケージと混ざって「importは通るが実行で落ちる」
# - transformers / trl / unsloth の相性不一致で、謎エラーや速度低下が起きる
#
# ※このセルは“おまじない”ではなく「再現性のための重要工程」です。

#!pip -q uninstall -y numpy pandas datasets trl transformers accelerate peft unsloth unsloth-zoo bitsandbytes xformers
!uv pip install "numpy==2.0.2" "pandas==2.2.2"

# unsloth-zoo が要求する範囲に合わせる
# ここでは Unsloth と相性の良い transformers / trl / accelerate / peft / bitsandbytes を固定します。
# 特に transformers は細かいバージョン差で挙動が変わりやすいので固定が重要です。
!uv pip install \
  "datasets==4.3.0" \
  "trl==0.24.0" \
  "transformers==4.56.2" \
  "accelerate==1.4.0" \
  "peft==0.13.2" \
  "bitsandbytes==0.45.0"

# unsloth / zoo を同系列で揃える（zoo側の要求に合わせる）
# Unsloth本体と unsloth-zoo は“セット運用”が基本です。片方だけ上げると壊れがちです。
!uv pip install "unsloth-zoo==2025.12.7" "unsloth==2025.12.7"



Using Python 3.12.12 environment at: /usr
Audited 2 packages in 464ms
Using Python 3.12.12 environment at: /usr
Resolved 68 packages in 490ms
Uninstalled 2 packages in 58ms
Installed 2 packages in 33ms
 - bitsandbytes==0.49.1
 + bitsandbytes==0.45.0
 - peft==0.18.1
 + peft==0.13.2
Using Python 3.12.12 environment at: /usr
Resolved 86 packages in 247ms
Uninstalled 3 packages in 25ms
Installed 3 packages in 24ms
 - bitsandbytes==0.45.0
 + bitsandbytes==0.49.1
 - unsloth==2026.1.4
 + unsloth==2025.12.7
 - unsloth-zoo==2026.1.4
 + unsloth-zoo==2025.12.7


In [ ]:
# huggingface_hubを最新にして整合性を合わせる
!pip install --upgrade huggingface_hub unsloth


  Using cached huggingface_hub-1.4.1-py3-none-any.whl.metadata (13 kB)
  Using cached unsloth-2026.1.4-py3-none-any.whl.metadata (66 kB)
  Using cached unsloth_zoo-2026.1.4-py3-none-any.whl.metadata (32 kB)
  Using cached peft-0.18.1-py3-none-any.whl.metadata (14 kB)
Using cached unsloth-2026.1.4-py3-none-any.whl (405 kB)
Using cached peft-0.18.1-py3-none-any.whl (556 kB)
Using cached unsloth_zoo-2026.1.4-py3-none-any.whl (310 kB)
  Attempting uninstall: peft
    Found existing installation: peft 0.13.2
    Uninstalling peft-0.13.2:
      Successfully uninstalled peft-0.13.2
  Attempting uninstall: unsloth_zoo
    Found existing installation: unsloth_zoo 2025.12.7
    Uninstalling unsloth_zoo-2025.12.7:
      Successfully uninstalled unsloth_zoo-2025.12.7
  Attempting uninstall: unsloth
    Found existing installation: unsloth 2025.12.7
    Uninstalling unsloth-2025.12.7:
      Successfully uninstalled unsloth-2025.12.7


## DPO１回目

### データセットのロード

In [ ]:
from datasets import load_dataset

# 1. 保存したjsonlファイルを読み込む
# 先ほどのスクリプトと同じディレクトリにあることを想定しているよ
data_files = {
    "train": "/content/drive/MyDrive/LLM2026/main_competition/DPO_data/dpo_train.jsonl",
    "eval": "/content/drive/MyDrive/LLM2026/main_competition/DPO_data/dpo_train.jsonl"
}

# json形式でロード
dataset = load_dataset("json", data_files=data_files)

# 2. train と eval に切り分け
train_dataset = dataset["train"]
eval_dataset = dataset["eval"]

# 3. 中身の確認（デバッグ用：最初の1件を表示）
print("--- Train Dataset Sample ---")
print(train_dataset[0])
print(f"\nTrain size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")




Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

--- Train Dataset Sample ---
{'prompt': '<|im_start|>system\nYou are a helpful assistant. Please format your response as follows:\nApproach: <step-by-step reasoning>\nOutput: <final answer><|im_end|>\n<|im_start|>user\nGenerate lab result data in CSV format.<|im_end|>\n<|im_start|>assistant\n', 'chosen': 'Approach:\n1. Task: Create lab result in CSV\n2. Complexity: medium - 5-8 fields, 2-3 levels\n3. Format rules: header row, comma-separated, quoted fields with commas\n4. Structure: Organize data logically with appropriate nesting\n5. Populate fields with realistic example data based on the schema.\n\nOutput:\norder_date,panel,patient_mrn,result_id,results\r\n2026-01-01,CBC,MRN-595201,LAB-6843976,"[{""test"": ""Glucose"", ""value"": 112.1, ""unit"": ""mg/dL"", ""flag"": ""high""}, {""test"": ""WBC"", ""value"": 108."\r\n2026-01-13,Lipid,MRN-841114,LAB-2871982,"[{""test"": ""WBC"", ""value"": 159.6, ""unit"": ""g/dL"", ""flag"": ""high""}, {""test"": ""Glucose"", ""value"": 169.1"\r\n20

In [ ]:
# データを分割する (90% を学習に、10% を評価に)
split_dataset = special_dpo_dataset.train_test_split(test_size=0.1, seed=3407)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"📊 分割完了！")
print(f"学習用 (train_dataset): {len(train_dataset)} 件")
print(f"評価用 (eval_dataset) : {len(eval_dataset)} 件")

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN2')

In [ ]:

# -----------------------------
# 1) HF login (once)
# -----------------------------
# HF Hub上のデータセットを読むため、HuggingFaceにログインします。
#
from huggingface_hub import login
login(HF_TOKEN)  # Colab will prompt

### SFTモデルのロード＆学習



In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer
import torch
from peft import set_peft_model_state_dict
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from trl import DPOConfig, DPOTrainer

# --- 1. 事前パッチ ---
PatchDPOTrainer()

# --- 2. モデルロード ---
base_model_id = "Qwen/Qwen3-4B-Instruct-2507"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_id,
    max_seq_length = 2048,
    load_in_4bit = True,
    trust_remote_code = True,
)

# --- 3. LoRA層の追加 (r=16に固定) ---
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# --- 4. 0.73の重みを注入 (外科手術) ---
print("🔄 0.73の重みを注入中...")
adapter_weights = hf_hub_download(repo_id="satoyutaka/llm2025_main_0", filename="adapter_model.safetensors")
state_dict = load_file(adapter_weights)
set_peft_model_state_dict(model, state_dict)

# --- 5. DPO学習の設定 (極限まで負荷を下げる) ---
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None, # メモリ節約
    args = DPOConfig(
        per_device_train_batch_size = 1, # T4の限界に挑む
        gradient_accumulation_steps = 8,
        warmup_ratio = 0.1,
        num_train_epochs = 3,
        learning_rate = 5e-6,
        fp16 = True,
        bf16 = False, # T4はFalse
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs",
        beta = 0.1,
        report_to = "none",
        remove_unused_columns = False, # これを追加すると安定することがあるよ
    ),
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    tokenizer = tokenizer,
    max_length = 1024,
    max_prompt_length = 512,
)

print("🚀 さあ、今度こそ学習開始！")
dpo_trainer.train()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/torchao/quantization/quant_api.py:2525: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.


🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.7: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 160 | Num Epochs = 3 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
1,0.319400,-0.317307,-3.105900,0.875000,2.788593,-535.520386,-165.992493,-2.622679,-2.767516,0,0,0
2,0.085400,-1.058482,-3.864253,1.000000,2.805771,-671.520264,-165.631424,-2.724725,-3.143741,No Log,No Log,No Log
3,0.975600,-0.091949,-2.443081,0.625000,2.351132,-548.687622,-174.627655,-2.293697,-2.749176,No Log,No Log,No Log
4,0.325100,-0.896705,-3.957639,0.875000,3.060934,-733.484924,-131.518402,-2.265240,-2.621856,No Log,No Log,No Log
5,0.483400,0.514583,-3.223665,0.875000,3.738248,-735.546326,-152.135773,-2.093873,-2.520550,No Log,No Log,No Log
6,0.092300,0.135107,-3.526444,1.000000,3.661550,-743.436829,-142.397766,-2.769930,-2.795242,No Log,No Log,No Log
7,0.295400,0.876510,-3.053158,0.875000,3.929667,-647.223389,-165.536362,-2.581647,-2.788344,No Log,No Log,No Log
8,0.008100,2.824279,-4.436924,1.000000,7.261203,-995.637085,-152.788422,-1.788278,-2.446592,No Log,No Log,No Log
9,0.101000,0.286647,-4.510213,1.000000,4.796861,-470.196838,-135.415009,-2.617023,-3.012541,No Log,No Log,No Log
10,0.066600,0.913057,-4.080938,1.000000,4.993996,-502.930969,-163.923615,-2.915379,-3.333272,No Log,No Log,No Log


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
1,0.319400,-0.317307,-3.105900,0.875000,2.788593,-535.520386,-165.992493,-2.622679,-2.767516,0,0,0
2,0.085400,-1.058482,-3.864253,1.000000,2.805771,-671.520264,-165.631424,-2.724725,-3.143741,No Log,No Log,No Log
3,0.975600,-0.091949,-2.443081,0.625000,2.351132,-548.687622,-174.627655,-2.293697,-2.749176,No Log,No Log,No Log
4,0.325100,-0.896705,-3.957639,0.875000,3.060934,-733.484924,-131.518402,-2.265240,-2.621856,No Log,No Log,No Log
5,0.483400,0.514583,-3.223665,0.875000,3.738248,-735.546326,-152.135773,-2.093873,-2.520550,No Log,No Log,No Log
6,0.092300,0.135107,-3.526444,1.000000,3.661550,-743.436829,-142.397766,-2.769930,-2.795242,No Log,No Log,No Log
7,0.295400,0.876510,-3.053158,0.875000,3.929667,-647.223389,-165.536362,-2.581647,-2.788344,No Log,No Log,No Log
8,0.008100,2.824279,-4.436924,1.000000,7.261203,-995.637085,-152.788422,-1.788278,-2.446592,No Log,No Log,No Log
9,0.101000,0.286647,-4.510213,1.000000,4.796861,-470.196838,-135.415009,-2.617023,-3.012541,No Log,No Log,No Log
10,0.066600,0.913057,-4.080938,1.000000,4.993996,-502.930969,-163.923615,-2.915379,-3.333272,No Log,No Log,No Log


TrainOutput(global_step=60, training_loss=0.05530330327386158, metrics={'train_runtime': 1150.1067, 'train_samples_per_second': 0.417, 'train_steps_per_second': 0.052, 'total_flos': 0.0, 'train_loss': 0.05530330327386158, 'epoch': 3.0})

In [ ]:
# from unsloth import FastLanguageModel
# import torch
# from peft import set_peft_model_state_dict
# from huggingface_hub import hf_hub_download
# from safetensors.torch import load_file

# # 1. ベースモデルのロード
# base_model_id = "QwAen/Qwen3-4B-Instruct-2507"
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = base_model_id,
#     max_seq_length = 2048,
#     load_in_4bit = True,
#     trust_remote_code = True,
# )

# # 2. DPO用のLoRA層を定義
# # 【重要】0.73アダプターと同じ r=16, lora_alpha=16 に設定します
# model = FastLanguageModel.get_peft_model(
#     model,
#     r = 16,  # ここを16に修正！
#     target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
#     lora_alpha = 16, # alphaもrと同じにするのが一般的だよ
#     lora_dropout = 0,
#     bias = "none",
#     use_gradient_checkpointing = "unsloth",
#     random_state = 3407,
# )

# # 3. 0.73の重みを強引に流し込む（外科手術パート）
# print("🔄 0.73の重みをダウンロードして適用中...")
# adapter_weights = hf_hub_download(repo_id="satoyutaka/llm2025_main_0", filename="adapter_model.safetensors")

# # 重みファイルをロード
# state_dict = load_file(adapter_weights)

# # 重みを現在のモデル（r=16になったモデル）に注入
# # これでサイズが一致するのでエラーが出なくなります
# set_peft_model_state_dict(model, state_dict)

# print("✅ ついに地獄脱出！！ 0.73の能力(r=16)を注入してDPOの準備が完了したよ！")


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.7: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
🔄 0.73の重みをダウンロードして適用中...
✅ ついに地獄脱出！！ 0.73の能力(r=16)を注入してDPOの準備が完了したよ！


In [ ]:
from google.colab import userdata
userdata.get('secretName')

In [ ]:
# # 16bitで結合して保存（これが一番精度が落ちにくい）
# new_model_name = "/content/drive/MyDrive/LLM2026/main_competition/DPO/struct-eval-comp-dpo-final"
# path = "/content/drive/MyDrive/LLM2026/main_competition/DPO/"


# # ローカルに保存する場合
# model.save_pretrained_merged(new_model_name, tokenizer, save_method = "merged_16bit")

# # Hugging Faceに直接アップロードする場合（推奨）
# model.push_to_hub_merged(
#     f"your-username/{new_model_name}",
#     tokenizer,
#     save_method = "merged_16bit",
#     token = "your_hf_token"
# )

In [ ]:
import os

# 1. 保存パスの設定（struct-eval-comp プロジェクト用）
adapter_name = "struct-eval-comp-dpo-v1"
drive_save_path = f"/content/drive/MyDrive/LLM2026/main_competition/DPO/{adapter_name}"
hf_model_id = f"satoyutaka/{adapter_name}" # 自分のHFユーザー名に合わせてね

# 2. Google ドライブへ保存
print(f"💾 Googleドライブに保存中: {drive_save_path}")
os.makedirs(drive_save_path, exist_ok=True)
model.save_pretrained(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

# 3. Hugging Face へアップロード
print(f"🚀 Hugging Faceにアップロード中: {hf_model_id}")
try:
    model.push_to_hub(hf_model_id)
    tokenizer.push_to_hub(hf_model_id)
    print("✅ HFへのアップロードも完了したよ！")
except Exception as e:
    print(f"⚠️ HFアップロードでエラーが出たけど、ドライブには保存されてるよ: {e}")

print("\n✨ すべての保存工程が完了しました！お疲れ様、ゆーちゃ！")


💾 Googleドライブに保存中: /content/drive/MyDrive/LLM2026/main_competition/DPO/struct-eval-comp-dpo-v1
🚀 Hugging Faceにアップロード中: satoyutaka/struct-eval-comp-dpo-v1
⚠️ HFアップロードでエラーが出たけど、ドライブには保存されてるよ: (Request ID: Root=1-6983e6bc-3ebdc2933a5f5fa7338bb3c6;6eee2338-7315-4cfd-9edc-703f3a03373d)

403 Forbidden: You don't have the rights to create a model under the namespace "satoyutaka".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.

✨ すべての保存工程が完了しました！お疲れ様、ゆーちゃ！


### 推論してチェック

In [ ]:
from unsloth import FastLanguageModel
import torch

# 1. パスの設定
adapter_path = "/content/drive/MyDrive/LLM2026/main_competition/DPO/struct-eval-comp-dpo-v1"
merged_save_path = "/content/drive/MyDrive/LLM2026/main_competition/DPO/merged_model"

# 2. モデルとアダプターのロード
# ※DPO学習時と同じ設定でロードします
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = adapter_path, # ドライブのアダプターパス
    max_seq_length = 2048,
    load_in_4bit = True,       # 4bitでロード
    trust_remote_code = True,
)

# 3. 結合（Merge）して保存
# save_pretrained_merged を使うと、Unslothが綺麗に統合してくれます
print("🔄 ベースモデルとアダプターを結合中...（これには数分かかります）")

model.save_pretrained_merged(
    merged_save_path,
    tokenizer,
    save_method = "merged_16bit", # 精度を保つために16bitで結合
)

print(f"✅ 結合完了！保存先: {merged_save_path}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/torchao/quantization/quant_api.py:2525: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.


🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.7: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.08G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [01:22<00:00, 41.47s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:54<00:00, 87.41s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/LLM2026/main_competition/DPO/merged_model`
✅ 結合完了！保存先: /content/drive/MyDrive/LLM2026/main_competition/DPO/merged_model


In [ ]:
import json
from tqdm import tqdm
from unsloth import FastLanguageModel

# 1. 推論モードの設定（すでにロード済みならスキップOK）
FastLanguageModel.for_inference(model)

def evaluate_model(test_dataset):
    correct_content = 0
    perfect_format = 0
    total = len(test_dataset)

    print(f"🧐 {total}件のデータで検証開始...")

    for i in tqdm(range(total)):
        item = test_dataset[i]
        prompt_input = item["prompt"]  # DPOデータセット作成時のキー名に合わせてね
        ground_truth = item["chosen"]  # 正解データ

        # モデル推論
        inputs = tokenizer([prompt_input], return_tensors = "pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens = 512, do_sample = False)
        prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # 回答部分のみ抽出（学習時の区切り文字に合わせて調整してね）
        response = prediction.split("### Response:\n")[-1].strip()

        # --- 評価1: フォーマットチェック ---
        # Markdownの ```json や 挨拶が入っていないか
        is_clean_json = response.startswith("{") and response.endswith("}") and "```" not in response
        if is_clean_json:
            perfect_format += 1

        # --- 評価2: 内容のチェック ---
        # ここでは単純比較。必要に応じてJSONの中身を取り出して比較してね
        if response == ground_truth:
            correct_content += 1
        else:
            # 失敗例を1つだけ表示（デバッグ用）
            if i == 0:
                print(f"\n[Diff Example]\nExpected: {ground_truth}\nGot: {response}")

    # スコア算出
    format_rate = (perfect_format / total) * 100
    accuracy_rate = (correct_content / total) * 100

    print("\n" + "="*30)
    print(f"🏆 検証結果 (Project: struct-eval-comp)")
    print(f"フォーマット遵守率: {format_rate:.2f}%")
    print(f"内容一致率（正解率）: {accuracy_rate:.2f}%")
    print("="*30)

# 実行！
evaluate_model(eval_dataset)

🧐 160件のデータで検証開始...


  1%|          | 1/160 [01:27<3:51:33, 87.38s/it]


[Diff Example]
Expected: Approach:
1. Task: Create lab result in CSV
2. Complexity: medium - 5-8 fields, 2-3 levels
3. Format rules: header row, comma-separated, quoted fields with commas
4. Structure: Organize data logically with appropriate nesting
5. Populate fields with realistic example data based on the schema.

Output:
order_date,panel,patient_mrn,result_id,results
2026-01-01,CBC,MRN-595201,LAB-6843976,"[{""test"": ""Glucose"", ""value"": 112.1, ""unit"": ""mg/dL"", ""flag"": ""high""}, {""test"": ""WBC"", ""value"": 108."
2026-01-13,Lipid,MRN-841114,LAB-2871982,"[{""test"": ""WBC"", ""value"": 159.6, ""unit"": ""g/dL"", ""flag"": ""high""}, {""test"": ""Glucose"", ""value"": 169.1"
2026-01-06,Lipid,MRN-882965,LAB-2096703,"[{""test"": ""Glucose"", ""value"": 155.0, ""unit"": ""g/dL"", ""flag"": ""high""}, {""test"": ""WBC"", ""value"": 184.4"

Got: system
You are a helpful assistant. Please format your response as follows:
Approach: <step-by-step reasoning>
Output: <final answe

100%|██████████| 160/160 [1:15:53<00:00, 28.46s/it]


🏆 検証結果 (Project: struct-eval-comp)
フォーマット遵守率: 0.00%
内容一致率（正解率）: 0.00%


# DPO２回目

## データのロード

In [ ]:
from datasets import load_dataset

# 1. 保存したjsonlファイルを読み込む
# 先ほどのスクリプトと同じディレクトリにあることを想定しているよ
data_files = {
    "train": "/content/drive/MyDrive/LLM2026/main_competition/DPO_data/struct_special_dpo_train.jsonl",
    "eval": "/content/drive/MyDrive/LLM2026/main_competition/DPO_data/struct_special_dpo_eval.jsonl"
}

# json形式でロード
dataset = load_dataset("json", data_files=data_files)

# 2. train と eval に切り分け
train_dataset = dataset["train"]
eval_dataset = dataset["eval"]

# 3. 中身の確認（デバッグ用：最初の1件を表示）
print("--- Train Dataset Sample ---")
print(train_dataset[0])
print(f"\nTrain size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")




Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

--- Train Dataset Sample ---
{'prompt': '### Instruction:\n<|im_start|>system\nYou are a helpful assistant. Please format your response as follows:\nApproach: <step-by-step reasoning>\nOutput: <final answer><|im_end|>\n<|im_start|>user\nOutput news article as TOML.<|im_end|>\n<|im_start|>assistant\n\n\n### Response:\n', 'chosen': 'Approach:\n1. Task: Create news article in TOML\n2. Complexity: complex - 8-10 fields, 3-4 levels\n3. Format rules: key=value syntax, [sections], proper types\n4. Structure: Organize data logically with appropriate nesting\n5. Populate fields with realistic example data based on the schema.\n\nOutput:\narticle_id = "ART-5766974"\nheadline = "Decentralized maximized challenge"\nauthor = "Dr. Eric Miller MD"\npublished = "2026-01-04T16:46:57.191248"\nsection = "Business"\ncontent = "Western experience subject current contain whether. Sell college camera amount none. Outside experience somebody.\\nRead challenge go degree almost. Place number language positive s

## モデルのロード＆学習

1. beta を下げる (0.1 → 0.05)

beta は、モデルが「どれだけ正解と不正解の差を重く見るか」の指標。値を小さくすると、今回のような「Markdownがついている（rejected）」ことに対して、より厳しい罰則を与えるようになる。

2. learning_rate を下げる (5e-6 → 2e-6)

0.73（または0.76）の重みを注入して「追加」で学習する場合、学習率が高いとせっかく積み上げた推論力が壊れちゃう（破滅的忘却）ことがある。お作法だけを優しく教え込むために、少し低めに設定。

3. dataset の指定

さっきフォルダに保存した「抽出データ（JSON/CSV特化）」を読み込んで、datasetに渡す。

In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer
import torch
from peft import set_peft_model_state_dict
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from trl import DPOConfig, DPOTrainer

# --- 1. 事前パッチ ---
PatchDPOTrainer()

# --- 2. モデルロード ---
base_model_id = "Qwen/Qwen3-4B-Instruct-2507"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_id,
    max_seq_length = 2048,
    load_in_4bit = True,
    trust_remote_code = True,
)

# --- 3. LoRA層の追加 (r=16に固定) ---
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# --- 4. 0.73の重みを注入 (外科手術) ---
print("🔄 0.73の重みを注入中...")
adapter_weights = hf_hub_download(repo_id="satoyutaka/LLM2025_main_DPO", filename="adapter_model.safetensors")
state_dict = load_file(adapter_weights)
set_peft_model_state_dict(model, state_dict)

# --- 5. DPO学習の設定 (極限まで負荷を下げる) ---
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None, # メモリ節約
    args = DPOConfig(
        per_device_train_batch_size = 1, # T4の限界に挑む
        gradient_accumulation_steps = 8,
        warmup_ratio = 0.1,
        num_train_epochs = 3,
        learning_rate = 2e-6,
        fp16 = True,
        bf16 = False, # T4はFalse
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs",
        beta = 0.05,
        report_to = "none",
        remove_unused_columns = False, # これを追加すると安定することがあるよ
    ),
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    tokenizer = tokenizer,
    max_length = 1024,
    max_prompt_length = 512,
)

print("🚀 さあ、学習開始！")
dpo_trainer.train()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/torchao/quantization/quant_api.py:2525: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.


🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.7: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

...

🚀 さあ、学習開始！


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 225 | Num Epochs = 3 | Total steps = 87
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
1,0.031500,2.148380,-1.510685,1.000000,3.659065,-649.437500,-144.202271,-2.254891,-2.814649,0,0,0
2,0.030100,2.191587,-1.445774,1.000000,3.637361,-819.227783,-156.341705,-2.083097,-2.589989,No Log,No Log,No Log
3,0.066200,0.780488,-2.159462,1.000000,2.939950,-330.452972,-137.961929,-3.368269,-3.170538,No Log,No Log,No Log
4,0.049500,1.565533,-2.039608,1.000000,3.605141,-522.885010,-143.530838,-2.429202,-2.723368,No Log,No Log,No Log
5,0.044700,1.785968,-1.962580,1.000000,3.748549,-558.618835,-190.077652,-2.159002,-2.825915,No Log,No Log,No Log
6,0.025700,1.947388,-2.063122,1.000000,4.010510,-807.451355,-209.683151,-2.570286,-2.988402,No Log,No Log,No Log
7,0.030600,2.099926,-1.854906,1.000000,3.954833,-587.823303,-191.816177,-2.120297,-2.676934,No Log,No Log,No Log
8,0.064900,1.753198,-1.270666,1.000000,3.023864,-677.984985,-199.541855,-1.977641,-2.559711,No Log,No Log,No Log
9,0.080300,1.704056,-1.670348,1.000000,3.374404,-614.062683,-190.855469,-2.792927,-2.932658,No Log,No Log,No Log
10,0.049900,1.807196,-1.727264,1.000000,3.534461,-876.006592,-221.278397,-2.666380,-2.838553,No Log,No Log,No Log


TrainOutput(global_step=87, training_loss=0.018866628156867862, metrics={'train_runtime': 1752.8522, 'train_samples_per_second': 0.385, 'train_steps_per_second': 0.05, 'total_flos': 0.0, 'train_loss': 0.018866628156867862, 'epoch': 3.0})

## モデルの保存

In [ ]:
import os

# 1. 保存パスの設定（struct-eval-comp プロジェクト用）
adapter_name = "struct-eval-comp-dpo-v2"
drive_save_path = f"/content/drive/MyDrive/LLM2026/main_competition/DPO/{adapter_name}"
hf_model_id = "satoyutaka/LLM2025_main_0_DPO2" # 自分のHFユーザー名に合わせてね

# 2. Google ドライブへ保存
print(f"💾 Googleドライブに保存中: {drive_save_path}")
os.makedirs(drive_save_path, exist_ok=True)
model.save_pretrained(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

# 3. Hugging Face へアップロード
print(f"🚀 Hugging Faceにアップロード中: {hf_model_id}")
try:
    model.push_to_hub(hf_model_id)
    tokenizer.push_to_hub(hf_model_id)
    print("✅ HFへのアップロードも完了したよ！")
except Exception as e:
    print(f"⚠️ HFアップロードでエラーが出たけど、ドライブには保存されてるよ: {e}")

print("\n✨ すべての保存工程が完了しました！お疲れ様、ゆーちゃ！")


💾 Googleドライブに保存中: /content/drive/MyDrive/LLM2026/main_competition/DPO/struct-eval-comp-dpo-v2
🚀 Hugging Faceにアップロード中: satoyutaka/LLM2025_main_0_DPO2
⚠️ HFアップロードでエラーが出たけど、ドライブには保存されてるよ: (Request ID: Root=1-69851889-0c51bdd709ab70524878221d;a26a7f23-fad8-42f6-b606-baa0866077a4)

403 Forbidden: Forbidden: you must use a write token to upload to a repository..
Cannot access content at: https://huggingface.co/api/models/satoyutaka/LLM2025_main_0_DPO2/preupload/main.
Make sure your token has the correct permissions.

✨ すべての保存工程が完了しました！お疲れ様、ゆーちゃ！


# 3回目

## データのロード

In [ ]:
from datasets import load_dataset

# 1. 保存したjsonlファイルを読み込む
# 先ほどのスクリプトと同じディレクトリにあることを想定しているよ
data_files = {
    "train": "/content/drive/MyDrive/LLM2026/main_competition/DPO_data/struct_special_2_dpo_train.jsonl",
    "eval": "/content/drive/MyDrive/LLM2026/main_competition/DPO_data/struct_special_2_dpo_eval.jsonl"
}

# json形式でロード
dataset = load_dataset("json", data_files=data_files)

# 2. train と eval に切り分け
train_dataset = dataset["train"]
eval_dataset = dataset["eval"]

# 3. 中身の確認（デバッグ用：最初の1件を表示）
print("--- Train Dataset Sample ---")
print(train_dataset[0])
print(f"\nTrain size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")




--- Train Dataset Sample ---
{'prompt': '### Instruction:\n<|im_start|>system\nYou are a helpful assistant. Please format your response as follows:\nApproach: <step-by-step reasoning>\nOutput: <final answer><|im_end|>\n<|im_start|>user\nOutput news article as TOML.<|im_end|>\n<|im_start|>assistant\n\n\n### Response:\n', 'chosen': 'article_id = "ART-5766974"\nheadline = "Decentralized maximized challenge"\nauthor = "Dr. Eric Miller MD"\npublished = "2026-01-04T16:46:57.191248"\nsection = "Business"\ncontent = "Western experience subject current contain whether. Sell college camera amount none. Outside experience somebody.\\nRead challenge go degree almost. Place number language positive serve part high."\ntags = [ "wall", "reality", "while", "company", "every",]\nviews = 66879\nshares = 3461', 'rejected': 'Approach:  \n1. Understand the request: The user wants a news article formatted in TOML (Tom\'s Obvious, Minimal Language), a configuration file format.  \n2. Recognize that TOML is n

## モデルのロードおよび学習

In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer
import torch
from peft import set_peft_model_state_dict
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from trl import DPOConfig, DPOTrainer

# --- 1. 事前パッチ ---
PatchDPOTrainer()

# --- 2. モデルロード ---
base_model_id = "Qwen/Qwen3-4B-Instruct-2507"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_id,
    max_seq_length = 2048,
    load_in_4bit = True,
    trust_remote_code = True,
)

# --- 3. LoRA層の追加 (r=16に固定) ---
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# --- 4. 0.766の重みを注入 (外科手術) ---
print("🔄 0.766の重みを注入中...")
adapter_weights = hf_hub_download(repo_id="satoyutaka/LLM2025_main_0_DPO2", filename="adapter_model.safetensors")
state_dict = load_file(adapter_weights)
set_peft_model_state_dict(model, state_dict)

# --- 5. DPO学習の設定 (極限まで負荷を下げる) ---
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None, # メモリ節約
    args = DPOConfig(
        per_device_train_batch_size = 1, # T4の限界に挑む
        gradient_accumulation_steps = 8,
        warmup_ratio = 0.1,
        num_train_epochs = 3,
        learning_rate = 2e-6,
        fp16 = True,
        bf16 = False, # T4はFalse
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs",
        beta = 0.05,
        report_to = "none",
        remove_unused_columns = False, # これを追加すると安定することがあるよ
    ),
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    tokenizer = tokenizer,
    max_length = 1024,
    max_prompt_length = 512,
)

print("🚀 さあ、学習開始！")
dpo_trainer.train()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.1.4: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

🚀 さあ、学習開始！


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 225 | Num Epochs = 3 | Total steps = 87
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
1,0.013900,3.023956,-1.809170,1.000000,4.833127,-408.334076,-150.162552,-2.185037,-2.788810,0,0,0
2,0.010000,3.442615,-1.770761,1.000000,5.213376,-576.487976,-162.828430,-2.169860,-2.555531,No Log,No Log,No Log
3,0.045200,0.988536,-2.421660,1.000000,3.410197,-61.305481,-143.205887,-3.780055,-3.106328,No Log,No Log,No Log
4,0.032300,1.935030,-2.337946,1.000000,4.272975,-251.214325,-149.497589,-2.407541,-2.682222,No Log,No Log,No Log
5,0.021600,2.444119,-2.132850,1.000000,4.576969,-293.342834,-193.483047,-2.074339,-2.783588,No Log,No Log,No Log
6,0.013400,2.981761,-2.347220,1.000000,5.328982,-555.064758,-215.365128,-2.716016,-2.948306,No Log,No Log,No Log
7,0.027500,2.851452,-2.051770,1.000000,4.903222,-326.648529,-195.753448,-1.896298,-2.637124,No Log,No Log,No Log
8,0.022500,2.654669,-1.587287,1.000000,4.241957,-420.246857,-205.874298,-1.880785,-2.534595,No Log,No Log,No Log
9,0.060300,2.610308,-1.968038,1.000000,4.578346,-359.086731,-196.828217,-2.875595,-2.901022,No Log,No Log,No Log
10,0.024900,2.351052,-2.057765,1.000000,4.408817,-621.221497,-227.887543,-2.870718,-2.805061,No Log,No Log,No Log


TrainOutput(global_step=87, training_loss=0.018272562000643592, metrics={'train_runtime': 1653.0606, 'train_samples_per_second': 0.408, 'train_steps_per_second': 0.053, 'total_flos': 0.0, 'train_loss': 0.018272562000643592, 'epoch': 3.0})

## モデルの保存

In [ ]:
import os

# 1. 保存パスの設定（struct-eval-comp プロジェクト用）
adapter_name = "struct-eval-comp-dpo-v3"
drive_save_path = f"/content/drive/MyDrive/LLM2026/main_competition/DPO/{adapter_name}"
hf_model_id = "satoyutaka/LLM2025_main_0_DPO3" # 自分のHFユーザー名に合わせてね

# 2. Google ドライブへ保存
print(f"💾 Googleドライブに保存中: {drive_save_path}")
os.makedirs(drive_save_path, exist_ok=True)
model.save_pretrained(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

# 3. Hugging Face へアップロード
print(f"🚀 Hugging Faceにアップロード中: {hf_model_id}")
try:
    model.push_to_hub(hf_model_id)
    tokenizer.push_to_hub(hf_model_id)
    print("✅ HFへのアップロードも完了したよ！")
except Exception as e:
    print(f"⚠️ HFアップロードでエラーが出たけど、ドライブには保存されてるよ: {e}")

print("\n✨ すべての保存工程が完了しました！お疲れ様、ゆーちゃ！")


💾 Googleドライブに保存中: /content/drive/MyDrive/LLM2026/main_competition/DPO/struct-eval-comp-dpo-v3
🚀 Hugging Faceにアップロード中: satoyutaka/LLM2025_main_0_DPO3
⚠️ HFアップロードでエラーが出たけど、ドライブには保存されてるよ: (Request ID: Root=1-69856a0b-7649f4ef0017d1b804a68edc;c6b945e2-7af5-45ef-ac20-d687212657c8)

403 Forbidden: You don't have the rights to create a model under the namespace "satoyutaka".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.

✨ すべての保存工程が完了しました！お疲れ様、ゆーちゃ！


# 4回目

🕵️‍♂️ 推論結果（0.77064）の分析レポート

ゆーちゃの読み通り、件数でおそらく30〜35件ほどミスをしている計算になるね。 でも、このミスの原因は「頭が悪いから（知識がないから）」じゃない。**「お行儀が悪いから」**だ！

ファイルの冒頭（p_7b33... のタスク）を見てみて：

JSON
"generation": "```json\n{\n  \"ecosystem\": {\n    \"name\": ...\n  }\n}\n```"
まだ ```json（Markdown）が出ちゃってる！！！

評価システム（採点プログラム）は、おそらくこういう挙動をしているはずだよ：

Strict Mode: 「JSON形式で答えろ」と言ったのに、先頭に ``` がある？ → パースエラー（0点）

Loose Mode: もし多少許容する採点だとしても、余計な文字が含まれていると完全一致率が下がる。

CSVの方（Glimmerbeetleのタスク）は綺麗に ``` なしで出力できているね。だから点数が伸びて0.77までは行けたんだ。つまり、**「JSONの問題のときだけ、まだMarkdownの癖が抜けていない」**のが敗因だ！

🚀 0.8への確実な道：Markdown抹殺計画

0.77までは「思考をカット」して伸びた。 次は**「Markdown（```）を物理的に消滅させたデータ」**で学習させて、モデルに「バッククォートなんて文字は私の辞書にはない」と思わせる必要がある。

昨日提案しかけた**「Pro+版」をさらに強化した、「Markdown抹殺・最終決戦仕様」**のコードをここに用意したよ。

今夜のセットは、このコードでデータを作って、学習（できれば0.73のアダプタから再スタート）してみて！

## データのロード

In [ ]:
from datasets import load_dataset

# 1. 保存したjsonlファイルを読み込む
# 先ほどのスクリプトと同じディレクトリにあることを想定しているよ
data_files = {
    "train": "/content/drive/MyDrive/LLM2026/main_competition/DPO_data/struct_special_4_dpo_train.jsonl",
    "eval": "/content/drive/MyDrive/LLM2026/main_competition/DPO_data/struct_special_4_dpo_eval.jsonl"
}

# json形式でロード
dataset = load_dataset("json", data_files=data_files)

# 2. train と eval に切り分け
train_dataset = dataset["train"]
eval_dataset = dataset["eval"]

# 3. 中身の確認（デバッグ用：最初の1件を表示）
print("--- Train Dataset Sample ---")
print(train_dataset[0])
print(f"\nTrain size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")




Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

--- Train Dataset Sample ---
{'prompt': '### Instruction:\n<|im_start|>system\nYou are a helpful assistant. Please format your response as follows:\nApproach: <step-by-step reasoning>\nOutput: <final answer><|im_end|>\n<|im_start|>user\nProduce a TOML document for a financial report.<|im_end|>\n<|im_start|>assistant\n\n\n### Response:\n', 'chosen': 'company = "Lee-Frank"\nticker = "PNOZ"\nquarter = "Q4 2020"\nrevenue = 95126242\nexpenses = 36470273\nnet_income = 18995645\neps = 2.94\n\n[metrics]\nprofit_margin = 20.94\nroe = 10.92', 'rejected': 'Approach:  \n1. Understand the requirements: The task is to create a TOML (Tom\'s Obvious, Minimal Language) document for a financial report. TOML is a simple, human-readable configuration format often used for storing configuration files or structured data.  \n2. Structure the financial report with logical sections: A typical financial report includes key sections such as metadata (e.g., report title, date, version), financial data (e.g., reve

## モデルのロードおよび学習

In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer
import torch
from peft import set_peft_model_state_dict
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from trl import DPOConfig, DPOTrainer

# --- 1. 事前パッチ ---
PatchDPOTrainer()

# --- 2. モデルロード ---
base_model_id = "Qwen/Qwen3-4B-Instruct-2507"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_id,
    max_seq_length = 2048,
    load_in_4bit = True,
    trust_remote_code = True,
)

# --- 3. LoRA層の追加 (r=16に固定) ---
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# --- 4. 0.766の重みを注入 (外科手術) ---
print("🔄 0.766の重みを注入中...")
adapter_weights = hf_hub_download(repo_id="satoyutaka/LLM2025_main_0", filename="adapter_model.safetensors")
state_dict = load_file(adapter_weights)
set_peft_model_state_dict(model, state_dict)

# --- 5. DPO学習の設定 (極限まで負荷を下げる) ---
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None, # メモリ節約
    args = DPOConfig(
        per_device_train_batch_size = 1, # T4の限界に挑む
        gradient_accumulation_steps = 8,
        warmup_ratio = 0.1,
        num_train_epochs = 3,
        learning_rate = 2e-6,
        fp16 = True,
        bf16 = False, # T4はFalse
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs",
        beta = 0.05,
        report_to = "none",
        remove_unused_columns = False, # これを追加すると安定することがあるよ
    ),
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    tokenizer = tokenizer,
    max_length = 1024,
    max_prompt_length = 512,
)

print("🚀 さあ、学習開始！")
dpo_trainer.train()


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.1.4: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
🔄 0.766の重みを注入中...


Extracting prompt in train dataset (num_proc=4):   0%|          | 0/225 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=4):   0%|          | 0/225 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=4):   0%|          | 0/225 [00:00<?, ? examples/s]

Extracting prompt in eval dataset (num_proc=4):   0%|          | 0/25 [00:00<?, ? examples/s]

...

🚀 さあ、学習開始！


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 225 | Num Epochs = 3 | Total steps = 87
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
1,0.192900,0.535160,-1.329438,1.000000,1.864599,-661.902832,-186.165802,-2.152515,-2.839837,0,0,0
2,0.285300,0.388789,-1.221886,1.000000,1.610675,-637.525085,-139.197739,-2.426863,-2.512148,No Log,No Log,No Log
3,0.136800,0.417504,-1.719938,1.000000,2.137443,-174.119553,-182.868500,-3.867260,-3.212847,No Log,No Log,No Log
4,0.180700,0.578478,-1.573873,1.000000,2.152351,-138.166000,-134.641815,-3.206994,-2.971640,No Log,No Log,No Log
5,0.126700,0.697943,-1.779088,1.000000,2.477030,-393.586639,-187.076248,-2.555626,-2.820475,No Log,No Log,No Log
6,0.209900,0.534819,-1.506357,1.000000,2.041176,-497.093781,-194.044830,-2.655343,-3.046289,No Log,No Log,No Log
7,0.136800,0.564970,-1.648324,1.000000,2.213294,-462.261536,-175.520126,-1.775722,-2.488558,No Log,No Log,No Log
8,0.199600,0.468119,-1.160031,1.000000,1.628150,-566.307922,-188.661377,-2.101189,-2.671010,No Log,No Log,No Log
9,0.235100,0.607369,-1.411679,0.875000,2.019048,-435.103485,-191.396561,-3.045953,-2.963161,No Log,No Log,No Log
10,0.127200,0.448485,-1.623377,1.000000,2.071863,-742.328857,-181.398651,-2.748287,-2.665987,No Log,No Log,No Log


TrainOutput(global_step=87, training_loss=0.08182804721097152, metrics={'train_runtime': 1840.0739, 'train_samples_per_second': 0.367, 'train_steps_per_second': 0.047, 'total_flos': 0.0, 'train_loss': 0.08182804721097152, 'epoch': 3.0})

## モデルの保存

In [ ]:
import os

# 1. 保存パスの設定（struct-eval-comp プロジェクト用）
adapter_name = "struct-eval-comp-dpo-v4"
drive_save_path = f"/content/drive/MyDrive/LLM2026/main_competition/DPO/{adapter_name}"
hf_model_id = "satoyutaka/LLM2025_main_0_DPO4" # 自分のHFユーザー名に合わせてね

# 2. Google ドライブへ保存
print(f"💾 Googleドライブに保存中: {drive_save_path}")
os.makedirs(drive_save_path, exist_ok=True)
model.save_pretrained(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

# 3. Hugging Face へアップロード
print(f"🚀 Hugging Faceにアップロード中: {hf_model_id}")
try:
    model.push_to_hub(hf_model_id)
    tokenizer.push_to_hub(hf_model_id)
    print("✅ HFへのアップロードも完了したよ！")
except Exception as e:
    print(f"⚠️ HFアップロードでエラーが出たけど、ドライブには保存されてるよ: {e}")

print("\n✨ すべての保存工程が完了しました！お疲れ様、ゆーちゃ！")


💾 Googleドライブに保存中: /content/drive/MyDrive/LLM2026/main_competition/DPO/struct-eval-comp-dpo-v4
🚀 Hugging Faceにアップロード中: satoyutaka/LLM2025_main_0_DPO4
⚠️ HFアップロードでエラーが出たけど、ドライブには保存されてるよ: (Request ID: Root=1-6986642c-212a2b017a74c4a50da10be9;737739d0-85b9-4933-b0b9-9eed8246ef3b)

403 Forbidden: You don't have the rights to create a model under the namespace "satoyutaka".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.

✨ すべての保存工程が完了しました！お疲れ様、ゆーちゃ！


# ５回目

🚀 struct-eval-comp: 第5回DPO学習メモ
📊 1. 使用データについて

「Markdown＆お喋り完全抹殺ドリル（Ver.5）」

構造化データ特化: JSON/CSV/YAMLなどの構造化タスクを重点的に強化（全体の約70%を配分）。

思考過程（CoT）の完全カット: Chosen（正解）から「Approach:」や「Output:」といったタグを物理的に排除し、純粋なデータのみを抽出。

Markdown＆不要解説の抹殺: 0.77モデルの弱点だった「```json」などのコードブロック記号や、末尾の「このデータは〜」といった余計な一言を、正規表現とトリミングで完全に消去。

対比学習: RejectedにあえてMarkdownやお喋りを残すことで、「余計なことを書くと減点」というルールをモデルに強く学習させる。

🤖 2. 学習モデルと外科手術

ベースモデル: Qwen3-4B-Instruct-2507

重み注入: 前回の最高スコア個体である 「0.77064モデル」 のアダプター重みを外科手術（set_peft_model_state_dict）でロード。

狙い: 0.77モデルが持つ「高い正解精度」を維持したまま、出力形式（お作法）だけを完璧に矯正する。

⚙️ 3. 学習パラメーター（スパルタ設定）

Learning Rate: 1e-6

すでに賢いモデルを壊さないよう、極めて低速で「お作法」だけを微調整。

Beta: 0.1 (または 0.15)

通常(0.05)より厳しめに設定。「お喋り」や「Markdown」を出した瞬間に報酬がガッツリ下がるスパルタ仕様。

Epochs: 3

250件の精選データに対し、3周かけて「無言でデータを出す」美学を叩き込む。

Gradient Accumulation: 16

T4 GPUのメモリ制限の中で、学習の安定性を最大化。

✨ 4. 期待される効果

パースエラーの解消: JSON出力時のバッククォートが消えることで、採点システムでの0点評価を回避。

完全一致率の向上: 文末の余計な解説文がなくなることで、構造化データの純度100%を実現。

大台突破: これまでの微増（0.766 → 0.77）から、お作法の改善による 「0.80超え」 へのジャンプアップを狙う。

## データのロード

In [ ]:
from datasets import load_dataset

# 1. 保存したjsonlファイルを読み込む
# 先ほどのスクリプトと同じディレクトリにあることを想定しているよ
data_files = {
    "train": "/content/drive/MyDrive/LLM2026/main_competition/DPO_data/struct_special_5_dpo_train.jsonl",
    "eval": "/content/drive/MyDrive/LLM2026/main_competition/DPO_data/struct_special_5_dpo_eval.jsonl"
}

# json形式でロード
dataset = load_dataset("json", data_files=data_files)

# 2. train と eval に切り分け
train_dataset = dataset["train"]
eval_dataset = dataset["eval"]

# 3. 中身の確認（デバッグ用：最初の1件を表示）
print("--- Train Dataset Sample ---")
print(train_dataset[0])
print(f"\nTrain size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")




Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

--- Train Dataset Sample ---
{'prompt': '### Instruction:\n<|im_start|>system\nYou are a helpful assistant. Please format your response as follows:\nApproach: <step-by-step reasoning>\nOutput: <final answer><|im_end|>\n<|im_start|>user\nProduce a TOML document for a financial report.<|im_end|>\n<|im_start|>assistant\n\n\n### Response:\n', 'chosen': 'company = "Lee-Frank"\nticker = "PNOZ"\nquarter = "Q4 2020"\nrevenue = 95126242\nexpenses = 36470273\nnet_income = 18995645\neps = 2.94\n\n[metrics]\nprofit_margin = 20.94\nroe = 10.92', 'rejected': 'Approach:  \n1. Understand the requirements: The task is to create a TOML (Tom\'s Obvious, Minimal Language) document for a financial report. TOML is a simple, human-readable configuration format often used for storing configuration files or structured data.  \n2. Structure the financial report with logical sections: A typical financial report includes key sections such as metadata (e.g., report title, date, version), financial data (e.g., reve

## モデルのロードおよび学習

In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer
import torch
from peft import set_peft_model_state_dict
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from trl import DPOConfig, DPOTrainer

# --- 1. 事前パッチ ---
PatchDPOTrainer()

# --- 2. モデルロード ---
base_model_id = "Qwen/Qwen3-4B-Instruct-2507"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_id,
    max_seq_length = 2048,
    load_in_4bit = True,
    trust_remote_code = True,
)

# --- 3. LoRA層の追加 (r=16に固定) ---
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# --- 4. 0.77の重みを注入 (外科手術) ---
print("🔄 0.77の重みを注入中...")
adapter_weights = hf_hub_download(repo_id="satoyutaka/LLM2025_main_0_DPO3", filename="adapter_model.safetensors")
state_dict = load_file(adapter_weights)
set_peft_model_state_dict(model, state_dict)

# --- 5. DPO学習の設定 (極限まで負荷を下げる) ---
# --- 5. DPO学習の設定 ---
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,
    args = DPOConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16, # 8から16に増やして、より安定させてみたよ！
        warmup_ratio = 0.1,
        num_train_epochs = 3,            # 3周でしっかり教育！
        learning_rate = 1e-6,            # 0.77の知能を守るために少し優しく
        fp16 = True,
        bf16 = False,
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs_silent_08", # フォルダ名でやる気アップ！
        beta = 0.1,                      # 「お喋り厳禁！」のスパルタ設定✨
        report_to = "none",
        remove_unused_columns = False,
        max_steps = -1,                  # epoch数で管理
    ),
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    tokenizer = tokenizer,
    max_length = 1024,
    max_prompt_length = 512,
)


print("🚀 さあ、学習開始！DPO5")
dpo_trainer.train()


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.1.4: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
🔄 0.766の重みを注入中...


adapter_model.safetensors:   0%|          | 0.00/132M [00:00<?, ?B/s]

Extracting prompt in train dataset (num_proc=4):   0%|          | 0/225 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=4):   0%|          | 0/225 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=4):   0%|          | 0/225 [00:00<?, ? examples/s]

...

🚀 さあ、学習開始！DPO5


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 225 | Num Epochs = 3 | Total steps = 45
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
1,0.000200,6.437078,-5.024148,1.000000,11.461227,-503.998413,-187.410004,-2.220841,-2.568519,0,0,0
2,0.000600,3.822001,-5.932301,1.000000,9.754303,-127.882568,-185.140076,-3.247776,-2.950645,No Log,No Log,No Log
3,0.000100,7.018194,-5.347616,1.000000,12.365809,-377.923157,-211.182236,-2.405779,-2.799829,No Log,No Log,No Log
4,0.000400,6.089043,-5.046889,1.000000,11.135932,-455.302643,-204.476105,-1.795668,-2.482455,No Log,No Log,No Log
5,0.001300,5.869621,-5.602189,1.000000,11.471811,-540.578430,-212.068909,-2.735227,-2.752090,No Log,No Log,No Log
6,0.003700,6.700970,-5.106014,1.000000,11.806984,-465.527802,-226.082626,-2.010399,-2.502396,No Log,No Log,No Log
7,0.000300,4.959206,-5.778049,1.000000,10.737255,-196.728439,-210.894608,-2.928922,-2.871271,No Log,No Log,No Log
8,0.006900,6.445859,-4.815911,1.000000,11.261770,-213.451309,-264.743500,-1.523323,-2.450961,No Log,No Log,No Log
9,0.001000,4.535202,-5.658299,1.000000,10.193501,-141.080093,-234.276901,-2.817700,-2.860320,No Log,No Log,No Log
10,0.000400,5.662985,-5.316668,1.000000,10.979652,-332.662231,-160.965164,-2.032891,-2.322538,No Log,No Log,No Log


TrainOutput(global_step=45, training_loss=0.0010311688348439122, metrics={'train_runtime': 1801.555, 'train_samples_per_second': 0.375, 'train_steps_per_second': 0.025, 'total_flos': 0.0, 'train_loss': 0.0010311688348439122, 'epoch': 3.0})

## モデルの保存

In [ ]:
import os

# 1. 保存パスの設定（struct-eval-comp プロジェクト用）
adapter_name = "struct-eval-comp-dpo-v5"
drive_save_path = f"/content/drive/MyDrive/LLM2026/main_competition/DPO/{adapter_name}"
hf_model_id = "satoyutaka/LLM2025_main_0_DPO4" # 自分のHFユーザー名に合わせてね

# 2. Google ドライブへ保存
print(f"💾 Googleドライブに保存中: {drive_save_path}")
os.makedirs(drive_save_path, exist_ok=True)
model.save_pretrained(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

# 3. Hugging Face へアップロード
print(f"🚀 Hugging Faceにアップロード中: {hf_model_id}")
try:
    model.push_to_hub(hf_model_id)
    tokenizer.push_to_hub(hf_model_id)
    print("✅ HFへのアップロードも完了したよ！")
except Exception as e:
    print(f"⚠️ HFアップロードでエラーが出たけど、ドライブには保存されてるよ: {e}")

print("\n✨ すべての保存工程が完了しました！お疲れ様、ゆーちゃ！")


💾 Googleドライブに保存中: /content/drive/MyDrive/LLM2026/main_competition/DPO/struct-eval-comp-dpo-v5
🚀 Hugging Faceにアップロード中: satoyutaka/LLM2025_main_0_DPO4
⚠️ HFアップロードでエラーが出たけど、ドライブには保存されてるよ: (Request ID: Root=1-69867bc2-0d9f13840cf1c7e66f486a15;9c8ef2e0-db97-43a7-9acf-fc56d8224854)

403 Forbidden: You don't have the rights to create a model under the namespace "satoyutaka".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.

✨ すべての保存工程が完了しました！お疲れ様、ゆーちゃ！


# 6回目

【学習実行：知能維持型・高精度フォーマット矯正学習】

■ 戦略概要 現在の最高スコア個体である「0.77064モデル（DPO Ver.3）」をベースモデルとして採用し、外科手術的な重み注入（Weight Injection）を行った上で、最終的な微調整（Fine-Tuning）を実施する。

■ ハイパーパラメータの設定意図

Learning Rate (1e-6): 既存の高度な知識を破壊しないよう、極めて低学習率で「お作法（出力形式）」の調整に特化させる。

Beta (0.1): 形式的な誤り（Markdownの残存等）に対して、標準設定以上のペナルティを課すことで、より厳格な出力制御を実現する。

Epochs (1~2): 過学習による「知能の退行（バカ化）」を防ぐため、反復回数を最小限に留め、データ整形による「癖の修正」のみを目的とする。

■ 期待される成果 高い推論能力（0.77相当）を保持したまま、推論結果からMarkdownブロックを完全に排除することで、コンペティション採点システムにおける完全一致スコアの最大化、および0.80の大台突破を狙う。

## データのロード

In [ ]:
from datasets import load_dataset

# 1. 保存したjsonlファイルを読み込む
# 先ほどのスクリプトと同じディレクトリにあることを想定しているよ
data_files = {
    "train": "/content/drive/MyDrive/LLM2026/main_competition/DPO_data/struct_special_6_dpo_train.jsonl",
    "eval": "/content/drive/MyDrive/LLM2026/main_competition/DPO_data/struct_special_6_dpo_eval.jsonl"
}

# json形式でロード
dataset = load_dataset("json", data_files=data_files)

# 2. train と eval に切り分け
train_dataset = dataset["train"]
eval_dataset = dataset["eval"]

# 3. 中身の確認（デバッグ用：最初の1件を表示）
print("--- Train Dataset Sample ---")
print(train_dataset[0])
print(f"\nTrain size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")




--- Train Dataset Sample ---
{'prompt': '### Instruction:\n<|im_start|>system\nYou are a helpful assistant. Please format your response as follows:\nApproach: <step-by-step reasoning>\nOutput: <final answer><|im_end|>\n<|im_start|>user\nOutput prescription as XML.<|im_end|>\n<|im_start|>assistant\n\n\n### Response:\n', 'chosen': 'Approach:\n1. Task: Create prescription in XML\n2. Complexity: medium - 5-8 fields, 2-3 levels\n3. Format rules: well-formed tags, proper nesting, escaped special chars\n4. Structure: Organize data logically with appropriate nesting\n5. Populate fields with realistic example data based on the schema.\n\nOutput:\n<?xml version="1.0" encoding="UTF-8"?>\n<prescription>\n<rx_id>RX-4085457</rx_id>\n<patient_mrn>MRN-357420</patient_mrn>\n<prescriber>Paul Contreras</prescriber>\n<date>2026-01-01</date>\n<medications>\n<name>Lisinopril</name>\n<dosage>10mg</dosage>\n<frequency>BID</frequency>\n<quantity>30</quantity>\n</medications>\n<medications>\n<name>Metformin</na

## モデルのロードおよび学習

In [ ]:
# 1. ライブラリを最新に更新して、衝突を解消するよ！
!pip install --upgrade huggingface_hub unsloth

  Using cached huggingface_hub-1.4.1-py3-none-any.whl.metadata (13 kB)


In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer
import torch
from peft import set_peft_model_state_dict
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from trl import DPOConfig, DPOTrainer

# --- 1. 事前パッチ ---
PatchDPOTrainer()

# --- 2. モデルロード ---
base_model_id = "Qwen/Qwen3-4B-Instruct-2507"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_id,
    max_seq_length = 2048,
    load_in_4bit = True,
    trust_remote_code = True,
)

# --- 3. LoRA層の追加 (r=16に固定) ---
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# --- 4. 0.77の重みを注入 (外科手術) ---
print("🔄 0.77の重みを注入中...")
adapter_weights = hf_hub_download(repo_id="satoyutaka/LLM2025_main_0_DPO3", filename="adapter_model.safetensors")
state_dict = load_file(adapter_weights)
set_peft_model_state_dict(model, state_dict)

# --- 5. DPO学習の設定 (極限まで負荷を下げる) ---
# --- 5. DPO学習の設定 ---
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,
    args = DPOConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16, # 8から16に増やして、より安定させてみたよ！
        warmup_ratio = 0.1,
        num_train_epochs = 1,            # 3周でしっかり教育！
        learning_rate = 1e-6,            # 0.77の知能を守るために少し優しく
        fp16 = True,
        bf16 = False,
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs_silent_08", # フォルダ名でやる気アップ！
        beta = 0.1,                      # 「お喋り厳禁！」のスパルタ設定✨
        report_to = "none",
        remove_unused_columns = False,
        max_steps = -1,                  # epoch数で管理
    ),
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    tokenizer = tokenizer,
    max_length = 1024,
    max_prompt_length = 512,
)


print("🚀 さあ、学習開始！DPO6")
dpo_trainer.train()


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.1.4: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
🔄 0.77の重みを注入中...


Extracting prompt in train dataset (num_proc=4):   0%|          | 0/225 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=4):   0%|          | 0/225 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=4):   0%|          | 0/225 [00:00<?, ? examples/s]

Extracting prompt in eval dataset (num_proc=4):   0%|          | 0/25 [00:00<?, ? examples/s]

...

🚀 さあ、学習開始！DPO6


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 225 | Num Epochs = 1 | Total steps = 15
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
1,0.000000,8.556685,-5.401835,1.000000,13.958522,-495.475311,-228.634048,-2.684129,-2.934120,0,0,0
2,0.000000,10.224525,-4.565536,1.000000,14.790062,-572.522217,-306.506409,-2.324314,-2.672029,No Log,No Log,No Log
3,0.002200,8.936655,-5.633445,1.000000,14.570101,-584.348877,-276.851654,-2.517584,-2.733589,No Log,No Log,No Log
4,0.000000,9.303516,-5.371501,1.000000,14.675016,-551.814575,-177.022736,-2.001591,-2.605359,No Log,No Log,No Log
5,0.004100,9.191992,-4.611028,1.000000,13.803020,-486.327881,-245.223328,-2.137632,-2.565508,No Log,No Log,No Log
6,0.000000,7.586540,-5.400438,1.000000,12.986979,-475.734772,-136.611542,-2.730550,-2.730646,No Log,No Log,No Log
7,0.002600,8.239649,-5.072419,1.000000,13.312069,-418.145996,-250.348724,-2.443224,-2.737478,No Log,No Log,No Log
8,0.000000,8.076910,-5.338228,1.000000,13.415138,-434.637238,-196.300385,-2.287039,-2.568206,No Log,No Log,No Log
9,0.000100,8.583739,-4.950971,1.000000,13.534710,-617.979126,-172.270569,-2.417048,-2.813840,No Log,No Log,No Log
10,0.020100,7.821888,-4.112011,1.000000,11.933899,-528.091858,-372.931335,-2.505326,-2.706741,No Log,No Log,No Log


TrainOutput(global_step=15, training_loss=0.002907154139461454, metrics={'train_runtime': 578.9142, 'train_samples_per_second': 0.389, 'train_steps_per_second': 0.026, 'total_flos': 0.0, 'train_loss': 0.002907154139461454, 'epoch': 1.0})

## モデルの保存

In [ ]:
import os

# 1. 保存パスの設定（struct-eval-comp プロジェクト用）
adapter_name = "struct-eval-comp-dpo-v6"
drive_save_path = f"/content/drive/MyDrive/LLM2026/main_competition/DPO/{adapter_name}"
hf_model_id = "satoyutaka/LLM2025_main_0_DPO6" # 自分のHFユーザー名に合わせてね

# 2. Google ドライブへ保存
print(f"💾 Googleドライブに保存中: {drive_save_path}")
os.makedirs(drive_save_path, exist_ok=True)
model.save_pretrained(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

# 3. Hugging Face へアップロード
print(f"🚀 Hugging Faceにアップロード中: {hf_model_id}")
try:
    model.push_to_hub(hf_model_id)
    tokenizer.push_to_hub(hf_model_id)
    print("✅ HFへのアップロードも完了したよ！")
except Exception as e:
    print(f"⚠️ HFアップロードでエラーが出たけど、ドライブには保存されてるよ: {e}")

print("\n✨ すべての保存工程が完了しました！お疲れ様、ゆーちゃ！")


💾 Googleドライブに保存中: /content/drive/MyDrive/LLM2026/main_competition/DPO/struct-eval-comp-dpo-v6
🚀 Hugging Faceにアップロード中: satoyutaka/LLM2025_main_0_DPO6
⚠️ HFアップロードでエラーが出たけど、ドライブには保存されてるよ: (Request ID: Root=1-6986e74a-4e3d52e372dda76d2dfc2862;e11d70c2-800f-47b4-b481-088a36a83c6b)

403 Forbidden: You don't have the rights to create a model under the namespace "satoyutaka".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.

✨ すべての保存工程が完了しました！お疲れ様、ゆーちゃ！


# ７回目

In [ ]:
# 1. ライブラリを最新に更新して、衝突を解消するよ！
!pip install --upgrade huggingface_hub unsloth

  Using cached huggingface_hub-1.4.1-py3-none-any.whl.metadata (13 kB)
  Using cached unsloth-2026.1.4-py3-none-any.whl.metadata (66 kB)
  Using cached unsloth_zoo-2026.1.4-py3-none-any.whl.metadata (32 kB)
  Using cached peft-0.18.1-py3-none-any.whl.metadata (14 kB)
Using cached unsloth-2026.1.4-py3-none-any.whl (405 kB)
Using cached peft-0.18.1-py3-none-any.whl (556 kB)
Using cached unsloth_zoo-2026.1.4-py3-none-any.whl (310 kB)
  Attempting uninstall: peft
    Found existing installation: peft 0.13.2
    Uninstalling peft-0.13.2:
      Successfully uninstalled peft-0.13.2
  Attempting uninstall: unsloth_zoo
    Found existing installation: unsloth_zoo 2025.12.7
    Uninstalling unsloth_zoo-2025.12.7:
      Successfully uninstalled unsloth_zoo-2025.12.7
  Attempting uninstall: unsloth
    Found existing installation: unsloth 2025.12.7
    Uninstalling unsloth-2025.12.7:
      Successfully uninstalled unsloth-2025.12.7


🛠️ 学習プロセスと設定の解説（真面目版）
このセルでは、前回の最高スコア（0.77136）を記録したモデルをベースに、さらに精度を高めるための継続的DPO学習を実行しています。 「推論の正確さ」と「形式の美しさ」を両立させるために、以下の高度な手法を取り入れています。

1. 既存アダプターの「重み注入」（Weight Injection）

新しいLoRA層を作成した後、前回スコアの良かったアダプター重み（LLM2025_main_0_DPO3）をプログラム的に注入しています。これにより、ゼロから学習し直すのではなく、**「すでに0.77の知能を持った状態」**から学習をスタートでき、効率的にスコアを伸ばすことが可能です。

2. DPOハイパーパラメータの最適化

Beta = 0.1: Beta値は、モデルが「元のベースモデルからどれだけ離れて良いか」を制御します。0.1に設定することで、モデルが自分の本来の推論能力（ベースモデルの賢さ）を捨てすぎることなく、フォーマットの矯正に集中できるようバランスを取っています。

Learning Rate = 8e-7: 通常のSFTよりも極めて低い学習率を採用しています。これは、既存の高度な推論知識を「破壊」せず、Markdown装飾の排除といった「出力スタイルの微調整」のみを精密に行うための設定です。

3. 多重エポックによる徹底教育

前回の実験では、1エポック（学習データ1周）ではQwen特有の「Markdownコードブロックで回答を囲む」というバイアスを完全に抜けきることができませんでした。今回は3エポックの学習を行うことで、作成した高品質なデータセットのパターンを深く定着させ、装飾の完全排除を目指します。

4. 勾配累積（Gradient Accumulation）の活用

GPUメモリの制約上、一度に処理できるバッチサイズは「1」に制限されていますが、gradient_accumulation_steps = 16 を設定することで、内部的には16サンプル分の情報をまとめてから重みを更新しています。これにより、少ない計算資源でも、大規模なマシンで学習させた時のような安定した学習進捗を実現しています。

## データのロード

In [ ]:
from datasets import load_dataset

# 1. 保存したjsonlファイルを読み込む
# 先ほどのスクリプトと同じディレクトリにあることを想定しているよ
data_files = {
    "train": "/content/drive/MyDrive/LLM2026/main_competition/DPO_data/struct_special_7_dpo_train.jsonl",
    "eval": "/content/drive/MyDrive/LLM2026/main_competition/DPO_data/struct_special_7_dpo_eval.jsonl"
}

# json形式でロード
dataset = load_dataset("json", data_files=data_files)

# 2. train と eval に切り分け
train_dataset = dataset["train"]
eval_dataset = dataset["eval"]

# 3. 中身の確認（デバッグ用：最初の1件を表示）
print("--- Train Dataset Sample ---")
print(train_dataset[0])
print(f"\nTrain size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")




--- Train Dataset Sample ---
{'prompt': '### Instruction:\n<|im_start|>system\nYou are a helpful assistant. Please format your response as follows:\nApproach: <step-by-step reasoning>\nOutput: <final answer><|im_end|>\n<|im_start|>user\nOutput api specification as XML.<|im_end|>\n<|im_start|>assistant\n\n\n### Response:\n', 'chosen': 'Approach:\n1. Task: Create api specification in XML\n2. Complexity: medium - 5-8 fields, 2-3 levels\n3. Format rules: well-formed tags, proper nesting, escaped special chars\n4. Structure: Organize data logically with appropriate nesting\n5. Populate fields with realistic example data based on the schema.\n\nOutput:\n<?xml version="1.0" encoding="UTF-8"?>\n<api_specification>\n<openapi>3.0.0</openapi>\n<info>\n<title>Multi-lateral demand-driven matrices API</title>\n<version>1.0.0</version>\n</info>\n<paths>\n</speech>\n<get>\n<summary>According institution everybody bad listen nearly.</summary>\n<parameters>\n<name>param0</name>\n<in>query</in>\n<type>st

## モデルのロードおよび学習

In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer
import torch
from peft import set_peft_model_state_dict
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from trl import DPOConfig, DPOTrainer

# --- 1. 事前パッチ ---
PatchDPOTrainer()

# --- 2. モデルロード ---
base_model_id = "Qwen/Qwen3-4B-Instruct-2507"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_id,
    max_seq_length = 2048,
    load_in_4bit = True,
    trust_remote_code = True,
)

# --- 3. LoRA層の追加 ---
# Rank(r)を維持しつつ、学習効率を最大化するよ。
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# --- 4. 0.77の重みを注入 (外科手術) ---
# 前回の最高傑作(DPO3/v6)の重みをロードして、ここをスタート地点にするよ！
print("🔄 前回の最高傑作（0.77）の知能を注入中...")
adapter_weights = hf_hub_download(repo_id="satoyutaka/LLM2025_main_0_DPO6", filename="adapter_model.safetensors")
state_dict = load_file(adapter_weights)
set_peft_model_state_dict(model, state_dict)

# --- 5. DPO学習の設定（0.80突破用スパルタ設定） ---
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None, # UnslothならNoneでメモリ節約！
    args = DPOConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16, # バッチサイズを仮想的に増やして学習を安定させる
        warmup_ratio = 0.1,
        num_train_epochs = 3,            # 1周では抜けなかった癖を3周で徹底排除！
        learning_rate = 8e-7,            # 1e-6より少し慎重に、既存の知能を壊さない速度
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs_struct_eval_v7",
        beta = 0.1,                      # 推論能力を守るための適度な重み（KLダイバージェンス）
        report_to = "none",
        remove_unused_columns = False,
    ),
    train_dataset = train_dataset, # さっきBERTで作った精鋭データ
    eval_dataset = eval_dataset,   # 検証用データ
    tokenizer = tokenizer,
    max_length = 1024,
    max_prompt_length = 512,
)

print("🚀 0.80の壁を壊しにいこう！学習開始！")
dpo_trainer.train()

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.1.4: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
🔄 前回の最高傑作（0.77）の知能を注入中...
🚀 0.80の壁を壊しにいこう！学習開始！


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 225 | Num Epochs = 3 | Total steps = 45
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
1,0.000800,7.211056,-5.000068,1.000000,12.211123,-359.278015,-347.552917,-2.971925,-2.844898,0,0,0
2,0.097300,6.873958,-5.486998,0.937500,12.360955,-366.408020,-304.301453,-2.964073,-3.053395,No Log,No Log,No Log
3,0.000400,8.062398,-4.083043,1.000000,12.145440,-392.275238,-245.983459,-2.845351,-3.064332,No Log,No Log,No Log
4,0.000100,7.970026,-5.055075,1.000000,13.025103,-363.062653,-171.567307,-2.886744,-3.126084,No Log,No Log,No Log
5,0.000300,7.416950,-4.190283,1.000000,11.607233,-379.061615,-187.661545,-2.710716,-3.171102,No Log,No Log,No Log
6,0.000700,7.724242,-4.688747,1.000000,12.412991,-408.433472,-193.311050,-2.836488,-3.143225,No Log,No Log,No Log
7,0.000000,8.325439,-5.657936,1.000000,13.983376,-381.006195,-159.326965,-3.013350,-3.032753,No Log,No Log,No Log
8,0.000000,8.082983,-6.609953,1.000000,14.692936,-362.859650,-185.332535,-2.950567,-2.830064,No Log,No Log,No Log
9,0.000000,8.195125,-5.938422,1.000000,14.133547,-391.007721,-162.332336,-3.123056,-3.103160,No Log,No Log,No Log
10,0.000000,8.696982,-5.346321,1.000000,14.043303,-387.392639,-158.034180,-2.914154,-2.958437,No Log,No Log,No Log


TrainOutput(global_step=45, training_loss=0.006161588863728361, metrics={'train_runtime': 2083.1808, 'train_samples_per_second': 0.324, 'train_steps_per_second': 0.022, 'total_flos': 0.0, 'train_loss': 0.006161588863728361, 'epoch': 3.0})

## モデルの保存

In [ ]:
import os

# 1. 保存パスの設定（struct-eval-comp プロジェクト用）
adapter_name = "struct-eval-comp-dpo-v7"
drive_save_path = f"/content/drive/MyDrive/LLM2026/main_competition/DPO/{adapter_name}"
hf_model_id = "satoyutaka/LLM2025_main_0_DPO6" # 自分のHFユーザー名に合わせてね

# 2. Google ドライブへ保存
print(f"💾 Googleドライブに保存中: {drive_save_path}")
os.makedirs(drive_save_path, exist_ok=True)
model.save_pretrained(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

# 3. Hugging Face へアップロード
print(f"🚀 Hugging Faceにアップロード中: {hf_model_id}")
try:
    model.push_to_hub(hf_model_id)
    tokenizer.push_to_hub(hf_model_id)
    print("✅ HFへのアップロードも完了したよ！")
except Exception as e:
    print(f"⚠️ HFアップロードでエラーが出たけど、ドライブには保存されてるよ: {e}")

print("\n✨ すべての保存工程が完了しました！お疲れ様、ゆーちゃ！")


💾 Googleドライブに保存中: /content/drive/MyDrive/LLM2026/main_competition/DPO/struct-eval-comp-dpo-v7
🚀 Hugging Faceにアップロード中: satoyutaka/LLM2025_main_0_DPO6
⚠️ HFアップロードでエラーが出たけど、ドライブには保存されてるよ: (Request ID: Root=1-698728ed-0cad97e923b360f40bbca3ba;99b16a72-6080-4924-ba5f-8c66558d0c77)

403 Forbidden: Forbidden: you must use a write token to upload to a repository..
Cannot access content at: https://huggingface.co/api/models/satoyutaka/LLM2025_main_0_DPO6/preupload/main.
Make sure your token has the correct permissions.

✨ すべての保存工程が完了しました！お疲れ様、ゆーちゃ！


# 8回目

In [ ]:
# 1. ライブラリを最新に更新して、衝突を解消するよ！
!pip install --upgrade huggingface_hub unsloth

  Using cached huggingface_hub-1.4.1-py3-none-any.whl.metadata (13 kB)


🚀 DPO学習：V8 スタイル矯正・最終フェーズ（スパルタ設定）
このセルでは、前回の DPO V7 で到達した最高スコア（0.77136）をベースに、出力フォーマットの完全性を追求するための**追加の微調整（Fine-tuning）**を実行します。

1. V8 の戦略的背景

V7 の分析により、モデルは「思考プロセス（Approach）」と「正しい抽出能力」を高いレベルで両立させていることが確認されました。しかし、ベースモデル（Qwen）が持つ「回答を Markdown コードブロックで囲む」という強力な先天的バイアスが、依然として数点のスコアロスを招いています。 V8 では、データの質はそのままに、**学習の「厳格さ」**を一段階引き上げることで、このバイアスを打破します。

2. ハイパーパラメータの攻撃的最適化

Beta = 0.01（極限設定）: Beta 値を V7（0.1）の 10 分の 1 まで下げています。これにより、モデルが「ベースモデルの元の傾向（Markdown 囲い）」に従おうとする制約を弱め、「Markdown なしの生データ出力」という今回の選好データに 100% 同期するように仕向けます。

Learning Rate = 2e-6: すでに学習済みの概念を定着させるため、V7 よりわずかに高い学習率を設定し、短期間（1 エポック）でスタイル変更を強力に焼き付けます。

3. 継続学習（Continued Training）のメリット

V7 の重みをスタート地点とすることで、ゼロから学習するリスクを避けつつ、**「知能は 0.77 以上のまま、見た目だけを矯正する」**という、コンペティションにおいて最も効率的な最適化を狙っています。

4. 期待される効果

この学習を完走させることで、推論結果からパースエラー（不必要な ```json 等）を完全に排除し、0.80 の壁を突破するクリーンな構造化データ出力を実現します。

## データのロード

In [ ]:
from datasets import load_dataset

# 1. 保存したjsonlファイルを読み込む
# 先ほどのスクリプトと同じディレクトリにあることを想定しているよ
data_files = {
    "train": "/content/drive/MyDrive/LLM2026/main_competition/DPO_data/struct_special_7_dpo_train.jsonl",
    "eval": "/content/drive/MyDrive/LLM2026/main_competition/DPO_data/struct_special_7_dpo_eval.jsonl"
}

# json形式でロード
dataset = load_dataset("json", data_files=data_files)

# 2. train と eval に切り分け
train_dataset = dataset["train"]
eval_dataset = dataset["eval"]

# 3. 中身の確認（デバッグ用：最初の1件を表示）
print("--- Train Dataset Sample ---")
print(train_dataset[0])
print(f"\nTrain size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")




--- Train Dataset Sample ---
{'prompt': '### Instruction:\n<|im_start|>system\nYou are a helpful assistant. Please format your response as follows:\nApproach: <step-by-step reasoning>\nOutput: <final answer><|im_end|>\n<|im_start|>user\nOutput api specification as XML.<|im_end|>\n<|im_start|>assistant\n\n\n### Response:\n', 'chosen': 'Approach:\n1. Task: Create api specification in XML\n2. Complexity: medium - 5-8 fields, 2-3 levels\n3. Format rules: well-formed tags, proper nesting, escaped special chars\n4. Structure: Organize data logically with appropriate nesting\n5. Populate fields with realistic example data based on the schema.\n\nOutput:\n<?xml version="1.0" encoding="UTF-8"?>\n<api_specification>\n<openapi>3.0.0</openapi>\n<info>\n<title>Multi-lateral demand-driven matrices API</title>\n<version>1.0.0</version>\n</info>\n<paths>\n</speech>\n<get>\n<summary>According institution everybody bad listen nearly.</summary>\n<parameters>\n<name>param0</name>\n<in>query</in>\n<type>st

## モデルのロードおよび学習

In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer
import torch
from peft import set_peft_model_state_dict
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from trl import DPOConfig, DPOTrainer

# --- 1. 事前パッチ ---
PatchDPOTrainer()

# --- 2. モデルロード ---
base_model_id = "Qwen/Qwen3-4B-Instruct-2507"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_id,
    max_seq_length = 2048,
    load_in_4bit = True,
    trust_remote_code = True,
)

# --- 3. LoRA層の追加 ---
# Rank(r)を維持しつつ、学習効率を最大化するよ。
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# --- 4. 0.77の重みを注入 (外科手術) ---
# 前回の最高傑作(DPO3/v6)の重みをロードして、ここをスタート地点にするよ！
print("🔄 前回の最高傑作（0.77）の知能を注入中...")
adapter_weights = hf_hub_download(repo_id="satoyutaka/LLM2025_main_0_DPO7", filename="adapter_model.safetensors")
state_dict = load_file(adapter_weights)
set_peft_model_state_dict(model, state_dict)

## --- V8: スパルタ追い込み設定 ---
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,
    args = DPOConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        warmup_ratio = 0.1,
        num_train_epochs = 1,            # すでに覚えてるデータだから1〜2周で十分！
        learning_rate = 2e-6,            # 最後にグイッと引き上げる
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs_V8_final_push",
        beta = 0.01,                     # 0.1から0.01へ。Qwenの「囲い癖」を力ずくで剥がす設定
        report_to = "none",
        remove_unused_columns = False,
    ),
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    tokenizer = tokenizer,
    max_length = 1024,
    max_prompt_length = 512,
)

print("🚀 0.80の壁を壊しにいこう！学習開始！")
dpo_trainer.train()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.1.4: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

🚀 0.80の壁を壊しにいこう！学習開始！


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 225 | Num Epochs = 1 | Total steps = 15
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
1,0.262100,0.739599,-0.504632,1.000000,1.244231,-357.428711,-348.015503,-2.968654,-2.841976,0,0,0
2,0.265700,0.706390,-0.554901,0.937500,1.261291,-364.508636,-304.921600,-2.962005,-3.051436,No Log,No Log,No Log
3,0.264700,0.831388,-0.412527,1.000000,1.243915,-389.760468,-246.405762,-2.842412,-3.060549,No Log,No Log,No Log
4,0.236700,0.831972,-0.512316,1.000000,1.344287,-359.565735,-172.248108,-2.880841,-3.119802,No Log,No Log,No Log
5,0.269900,0.785060,-0.429276,1.000000,1.214336,-374.725159,-188.686279,-2.703829,-3.163083,No Log,No Log,No Log
6,0.249700,0.826622,-0.478201,1.000000,1.304823,-403.013672,-194.243698,-2.830805,-3.134671,No Log,No Log,No Log
7,0.208200,0.897582,-0.582827,1.000000,1.480409,-374.502319,-161.030289,-3.006246,-3.020973,No Log,No Log,No Log
8,0.197600,0.875776,-0.681757,1.000000,1.557533,-356.111847,-187.408661,-2.936760,-2.819448,No Log,No Log,No Log
9,0.202700,0.899093,-0.616137,1.000000,1.515230,-383.049683,-164.561768,-3.110542,-3.087013,No Log,No Log,No Log
10,0.203000,0.951657,-0.556704,1.000000,1.508361,-379.196808,-160.241409,-2.903318,-2.943004,No Log,No Log,No Log


TrainOutput(global_step=15, training_loss=0.23566920459270477, metrics={'train_runtime': 761.194, 'train_samples_per_second': 0.296, 'train_steps_per_second': 0.02, 'total_flos': 0.0, 'train_loss': 0.23566920459270477, 'epoch': 1.0})

## モデルの保存

In [ ]:
import os

# 1. 保存パスの設定（struct-eval-comp プロジェクト用）
adapter_name = "struct-eval-comp-dpo-v8"
drive_save_path = f"/content/drive/MyDrive/LLM2026/main_competition/DPO/{adapter_name}"
hf_model_id = "satoyutaka/LLM2025_main_0_DPO8" # 自分のHFユーザー名に合わせてね

# 2. Google ドライブへ保存
print(f"💾 Googleドライブに保存中: {drive_save_path}")
os.makedirs(drive_save_path, exist_ok=True)
model.save_pretrained(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

# 3. Hugging Face へアップロード
print(f"🚀 Hugging Faceにアップロード中: {hf_model_id}")
try:
    model.push_to_hub(hf_model_id)
    tokenizer.push_to_hub(hf_model_id)
    print("✅ HFへのアップロードも完了したよ！")
except Exception as e:
    print(f"⚠️ HFアップロードでエラーが出たけど、ドライブには保存されてるよ: {e}")

print("\n✨ すべての保存工程が完了しました！お疲れ様、ゆーちゃ！")


💾 Googleドライブに保存中: /content/drive/MyDrive/LLM2026/main_competition/DPO/struct-eval-comp-dpo-v8
🚀 Hugging Faceにアップロード中: satoyutaka/LLM2025_main_0_DPO8


README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

⚠️ HFアップロードでエラーが出たけど、ドライブには保存されてるよ: (Request ID: Root=1-69877859-5c7882f974eaea502e3838db;c95756be-5f47-4571-b6d6-fc1133941683)

403 Forbidden: Forbidden: you must use a write token to upload to a repository..
Cannot access content at: https://huggingface.co/api/models/satoyutaka/LLM2025_main_0_DPO8/preupload/main.
Make sure your token has the correct permissions.

✨ すべての保存工程が完了しました！お疲れ様、ゆーちゃ！


# 9回目

In [ ]:
# 1. ライブラリを最新に更新して、衝突を解消するよ！
!pip install --upgrade huggingface_hub unsloth

  Using cached huggingface_hub-1.4.1-py3-none-any.whl.metadata (13 kB)


🚀 DPO学習：V9 構造知能の復元とスタイル最適化（中庸の設定）
このセルでは、前回の V8 で発生した「構造（Schema）の崩壊」という課題を解決し、最高スコア（0.77136）を超えるための**「バランス調整フェーズ」**を実行します。

1. V8 の分析と V9 の戦略

V8 では Beta 値を 0.01 まで下げた結果、Markdown 記号の排除には成功しましたが、引き換えにモデルが複雑なネスト構造を維持する能力（論理的整合性）を失い、スコアが 0.71 まで低下しました。 V9 では、**「知能を壊さずに、不要な装飾だけを削ぎ落とす」**ために、DPO の制約を緩和し、より安定した学習を目指します。

2. ハイパーパラメータの再最適化

Beta = 0.06（黄金比の探索）: V7（0.1：保守的）と V8（0.01：過激）の中間にあたる 0.06 を採用します。これにより、Qwen 本来の「高い構造化能力」を呼び戻しつつ、Markdown 囲いを「好ましくないもの」として認識させる絶妙なバランスを維持します。

Learning Rate = 1e-6: 学習率を V7 の成功時と同じ水準に戻します。急激な変化を避け、モデルが「思考（Approach）」を伴いながら「生のデータ」を出力する習慣をじわじわと定着させます。

Num Epochs = 2: 1 エポックでは定着しきれなかった「マナー」を、2 エポックかけて丁寧に教育します。

3. 学習の期待値

V9 の狙いは、**「V7 の知能（0.77）」＋「V8 のクリーンな見た目」**の合体です。 思考プロセス（Approach）を飛ばさずに実行させ、かつ最終的な Output: セクションからバッククォートを排除することで、パースエラーによる失点を防ぎ、目標の 0.80 突破を狙います。

## データのロード

In [ ]:
from datasets import load_dataset

# 1. 保存したjsonlファイルを読み込む
# 先ほどのスクリプトと同じディレクトリにあることを想定しているよ
data_files = {
    "train": "/content/drive/MyDrive/LLM2026/main_competition/DPO_data/struct_special_7_dpo_train.jsonl",
    "eval": "/content/drive/MyDrive/LLM2026/main_competition/DPO_data/struct_special_7_dpo_eval.jsonl"
}

# json形式でロード
dataset = load_dataset("json", data_files=data_files)

# 2. train と eval に切り分け
train_dataset = dataset["train"]
eval_dataset = dataset["eval"]

# 3. 中身の確認（デバッグ用：最初の1件を表示）
print("--- Train Dataset Sample ---")
print(train_dataset[0])
print(f"\nTrain size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")




--- Train Dataset Sample ---
{'prompt': '### Instruction:\n<|im_start|>system\nYou are a helpful assistant. Please format your response as follows:\nApproach: <step-by-step reasoning>\nOutput: <final answer><|im_end|>\n<|im_start|>user\nOutput api specification as XML.<|im_end|>\n<|im_start|>assistant\n\n\n### Response:\n', 'chosen': 'Approach:\n1. Task: Create api specification in XML\n2. Complexity: medium - 5-8 fields, 2-3 levels\n3. Format rules: well-formed tags, proper nesting, escaped special chars\n4. Structure: Organize data logically with appropriate nesting\n5. Populate fields with realistic example data based on the schema.\n\nOutput:\n<?xml version="1.0" encoding="UTF-8"?>\n<api_specification>\n<openapi>3.0.0</openapi>\n<info>\n<title>Multi-lateral demand-driven matrices API</title>\n<version>1.0.0</version>\n</info>\n<paths>\n</speech>\n<get>\n<summary>According institution everybody bad listen nearly.</summary>\n<parameters>\n<name>param0</name>\n<in>query</in>\n<type>st

## モデルのロードおよび学習

In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer
import torch
from peft import set_peft_model_state_dict
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from trl import DPOConfig, DPOTrainer

# --- 1. 事前パッチ ---
PatchDPOTrainer()

# --- 2. モデルロード ---
base_model_id = "Qwen/Qwen3-4B-Instruct-2507"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_id,
    max_seq_length = 2048,
    load_in_4bit = True,
    trust_remote_code = True,
)

# --- 3. LoRA層の追加 ---
# Rank(r)を維持しつつ、学習効率を最大化するよ。
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# --- 4. 0.77の重みを注入 (外科手術) ---
# 前回の最高傑作(DPO3/v6)の重みをロードして、ここをスタート地点にするよ！
print("🔄 前回の最高傑作（0.77）の知能を注入中...")
adapter_weights = hf_hub_download(repo_id="satoyutaka/LLM2025_main_0_DPO7", filename="adapter_model.safetensors")
state_dict = load_file(adapter_weights)
set_peft_model_state_dict(model, state_dict)

## --- V8: スパルタ追い込み設定 ---
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,
    args = DPOConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        warmup_ratio = 0.1,
        num_train_epochs = 1,            # スタイルを覚えるために２回
        learning_rate = 1e-6,            # V7の成功した学習率に戻し、じわじわと定着させる。
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs_V8_final_push",
        beta = 0.06,                     # 0.01は「極端」すぎた。少し余裕を持たせて知能を呼び戻す。
        report_to = "none",
        remove_unused_columns = False,
    ),
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    tokenizer = tokenizer,
    max_length = 1024,
    max_prompt_length = 512,
)

print("🚀 0.80の壁を壊しにいこう！学習開始！")
dpo_trainer.train()

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.1.4: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
🔄 前回の最高傑作（0.77）の知能を注入中...
🚀 0.80の壁を壊しにいこう！学習開始！


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 225 | Num Epochs = 1 | Total steps = 15
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
1,0.004800,4.437593,-3.027795,1.000000,7.465387,-357.428711,-348.015503,-2.968654,-2.841976,0,0,0
2,0.056100,4.238339,-3.329407,0.937500,7.567745,-364.508636,-304.921600,-2.962005,-3.051436,No Log,No Log,No Log
3,0.004000,4.954006,-2.465780,1.000000,7.419785,-390.332458,-246.249359,-2.843127,-3.061379,No Log,No Log,No Log
4,0.001300,4.905714,-3.046215,1.000000,7.951929,-361.001007,-171.786789,-2.883746,-3.122933,No Log,No Log,No Log
5,0.004000,4.575350,-2.533083,1.000000,7.108433,-376.975281,-187.976761,-2.708060,-3.167612,No Log,No Log,No Log
6,0.004800,4.776915,-2.821721,1.000000,7.598636,-406.060669,-193.452271,-2.834999,-3.140115,No Log,No Log,No Log
7,0.000400,5.142111,-3.412731,1.000000,8.554842,-378.558746,-159.626480,-3.011655,-3.029099,No Log,No Log,No Log
8,0.000400,4.992089,-3.991629,1.000000,8.983717,-360.487946,-185.760132,-2.946853,-2.827185,No Log,No Log,No Log
9,0.000400,5.075625,-3.585903,1.000000,8.661529,-388.365234,-162.713150,-3.119660,-3.098666,No Log,No Log,No Log
10,0.000400,5.381866,-3.223860,1.000000,8.605725,-384.664703,-158.301971,-2.911673,-2.954408,No Log,No Log,No Log


TrainOutput(global_step=15, training_loss=0.006667492154520005, metrics={'train_runtime': 741.0348, 'train_samples_per_second': 0.304, 'train_steps_per_second': 0.02, 'total_flos': 0.0, 'train_loss': 0.006667492154520005, 'epoch': 1.0})

## モデルの保存

In [ ]:
import os

# 1. 保存パスの設定（struct-eval-comp プロジェクト用）
adapter_name = "struct-eval-comp-dpo-v9"
drive_save_path = f"/content/drive/MyDrive/LLM2026/main_competition/DPO/{adapter_name}"
hf_model_id = "satoyutaka/LLM2025_main_0_DPO9" # 自分のHFユーザー名に合わせてね

# 2. Google ドライブへ保存
print(f"💾 Googleドライブに保存中: {drive_save_path}")
os.makedirs(drive_save_path, exist_ok=True)
model.save_pretrained(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

# 3. Hugging Face へアップロード
print(f"🚀 Hugging Faceにアップロード中: {hf_model_id}")
try:
    model.push_to_hub(hf_model_id)
    tokenizer.push_to_hub(hf_model_id)
    print("✅ HFへのアップロードも完了したよ！")
except Exception as e:
    print(f"⚠️ HFアップロードでエラーが出たけど、ドライブには保存されてるよ: {e}")

print("\n✨ すべての保存工程が完了しました！お疲れ様、ゆーちゃ！")


💾 Googleドライブに保存中: /content/drive/MyDrive/LLM2026/main_competition/DPO/struct-eval-comp-dpo-v9
🚀 Hugging Faceにアップロード中: satoyutaka/LLM2025_main_0_DPO9
⚠️ HFアップロードでエラーが出たけど、ドライブには保存されてるよ: (Request ID: Root=1-69878670-6712afe36c53b19662adebe2;be8a581e-9aae-4bf4-8c79-e60f573e14f5)

403 Forbidden: You don't have the rights to create a model under the namespace "satoyutaka".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.

✨ すべての保存工程が完了しました！お疲れ様、ゆーちゃ！


# 10回目

In [ ]:
# 1. ライブラリを最新に更新して、衝突を解消するよ！
!pip install --upgrade huggingface_hub unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.7/405.7 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 kB 16.8 MB/s eta 0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.13.2
    Uninstalling peft-0.13.2:
      Successfully uninstalled peft-0.13.2
  Attempting uninstall: unsloth_zoo
    Found existing installation: unsloth_zoo 2025.12.7
    Uninstalling unsloth_zoo-2025.12.7:
      Successfully uninstalled unsloth_zoo-2025.12.7
  Attempting uninstall: unsloth
    Found existing installation: unsloth 2025.12.7
    Uninstalling unsloth-2025.12.7:
      Successfully uninstalled unsloth-2025.12.7


🚀 DPO学習：V10 最終調整フェーズ（知能と形式の黄金比）
このセルは、プロジェクト「struct-eval-comp」の集大成となる V10 学習プロセスです。 V8 (Beta 0.01) での構造崩壊と、V9 (Beta 0.06) での形式改善の停滞という極端な結果を分析し、「推論能力を一切削らずに、Markdownの癖だけを剥ぎ取る」 最適なパラメータを適用します。

1. V10 の戦略的ポジショニング

分析の結果、Qwen の構造化知能を維持しつつ Markdown バイアスを打破できる「スイートスポット」は Beta 値 0.03 〜 0.04 の間にあると推定されました。V10 ではその中心値である 0.035 を採用し、知能の安定と形式の矯正を両立させます。

2. ハイパーパラメータの最終最適化

Beta = 0.035: 0.01（過激）と 0.06（保守的）のギャップを埋める、本プロジェクトにおける「黄金比」設定です。

Learning Rate = 1e-6: V7/V9 での成功パターンを維持し、モデルの既存知能（0.77水準）を損なうことなく微調整を行います。

Num Epochs = 2: 2 回の反復により、1 回では剥がれきれなかった Markdown への依存を確実に排除します。

3. 期待される到達点

本学習の完走により、V7/V9 が持っていた「高い抽出精度」を保持したまま、V8 が見せた「クリーンな生データ出力」を実現し、スコア 0.80 の壁を突破することを最終目標とします。

## データのロード

In [ ]:
from datasets import load_dataset

# 1. 保存したjsonlファイルを読み込む
# 先ほどのスクリプトと同じディレクトリにあることを想定しているよ
data_files = {
    "train": "/content/drive/MyDrive/LLM2026/main_competition/DPO_data/struct_special_7_dpo_train.jsonl",
    "eval": "/content/drive/MyDrive/LLM2026/main_competition/DPO_data/struct_special_7_dpo_eval.jsonl"
}

# json形式でロード
dataset = load_dataset("json", data_files=data_files)

# 2. train と eval に切り分け
train_dataset = dataset["train"]
eval_dataset = dataset["eval"]

# 3. 中身の確認（デバッグ用：最初の1件を表示）
print("--- Train Dataset Sample ---")
print(train_dataset[0])
print(f"\nTrain size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")




--- Train Dataset Sample ---
{'prompt': '### Instruction:\n<|im_start|>system\nYou are a helpful assistant. Please format your response as follows:\nApproach: <step-by-step reasoning>\nOutput: <final answer><|im_end|>\n<|im_start|>user\nOutput api specification as XML.<|im_end|>\n<|im_start|>assistant\n\n\n### Response:\n', 'chosen': 'Approach:\n1. Task: Create api specification in XML\n2. Complexity: medium - 5-8 fields, 2-3 levels\n3. Format rules: well-formed tags, proper nesting, escaped special chars\n4. Structure: Organize data logically with appropriate nesting\n5. Populate fields with realistic example data based on the schema.\n\nOutput:\n<?xml version="1.0" encoding="UTF-8"?>\n<api_specification>\n<openapi>3.0.0</openapi>\n<info>\n<title>Multi-lateral demand-driven matrices API</title>\n<version>1.0.0</version>\n</info>\n<paths>\n</speech>\n<get>\n<summary>According institution everybody bad listen nearly.</summary>\n<parameters>\n<name>param0</name>\n<in>query</in>\n<type>st

## モデルのロードおよび学習

In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer
import torch
from peft import set_peft_model_state_dict
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from trl import DPOConfig, DPOTrainer

# --- 1. 事前パッチ ---
PatchDPOTrainer()

# --- 2. モデルロード ---
base_model_id = "Qwen/Qwen3-4B-Instruct-2507"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_id,
    max_seq_length = 2048,
    load_in_4bit = True,
    trust_remote_code = True,
)

# --- 3. LoRA層の追加 ---
# Rank(r)を維持しつつ、学習効率を最大化するよ。
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# --- 4. 0.77の重みを注入 (外科手術) ---
# 前回の最高傑作(DPO3/v6)の重みをロードして、ここをスタート地点にするよ！
print("🔄 前回の最高傑作（0.77）の知能を注入中...")
adapter_weights = hf_hub_download(repo_id="satoyutaka/LLM2025_main_0_DPO7", filename="adapter_model.safetensors")
state_dict = load_file(adapter_weights)
set_peft_model_state_dict(model, state_dict)

## --- V8: スパルタ追い込み設定 ---
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None, # Unslothの最適化を利用
    args = DPOConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        warmup_ratio = 0.1,
        num_train_epochs = 2,            # 2周で「マナー」を脳に定着させる
        learning_rate = 1e-6,            # 成功パターンの学習率を維持
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs_V10_final_gold",
        beta = 0.035,                    # 知能(0.06)と矯正(0.01)の究極の妥協点！
        report_to = "none",
        remove_unused_columns = False,
        max_length = 1024,
        max_prompt_length = 512,
    ),
    train_dataset = train_dataset, # BERT厳選データ
    eval_dataset = eval_dataset,
    tokenizer = tokenizer,
)

print("🚀 0.80の壁を壊しにいこう！学習開始！")
dpo_trainer.train()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.1.4: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

🚀 0.80の壁を壊しにいこう！学習開始！


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 225 | Num Epochs = 2 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
1,0.025800,2.588596,-1.766214,1.000000,4.354809,-357.428711,-348.015503,-2.968654,-2.841976,0,0,0
2,0.062100,2.472365,-1.942154,0.937500,4.414518,-364.508636,-304.921600,-2.962005,-3.051436,No Log,No Log,No Log
3,0.025900,2.891819,-1.440314,1.000000,4.332133,-390.275818,-246.304871,-2.843422,-3.062081,No Log,No Log,No Log
4,0.014500,2.863783,-1.780089,1.000000,4.643873,-360.940582,-171.876236,-2.883394,-3.122497,No Log,No Log,No Log
5,0.028200,2.679928,-1.479800,1.000000,4.159728,-376.661743,-188.038696,-2.707663,-3.167265,No Log,No Log,No Log
6,0.024800,2.803828,-1.650186,1.000000,4.454014,-405.566528,-193.571777,-2.834270,-3.139433,No Log,No Log,No Log
7,0.008500,3.028465,-1.997451,1.000000,5.025916,-377.733032,-159.817612,-3.010443,-3.027735,No Log,No Log,No Log
8,0.007900,2.945105,-2.336452,1.000000,5.281556,-359.543610,-185.988754,-2.944874,-2.826115,No Log,No Log,No Log
9,0.008100,3.002569,-2.100769,1.000000,5.103338,-387.171295,-162.970078,-3.117964,-3.096655,No Log,No Log,No Log
10,0.008100,3.187602,-1.891520,1.000000,5.079122,-383.288116,-158.614380,-2.909844,-2.952254,No Log,No Log,No Log


TrainOutput(global_step=30, training_loss=0.019361726505060992, metrics={'train_runtime': 1532.2621, 'train_samples_per_second': 0.294, 'train_steps_per_second': 0.02, 'total_flos': 0.0, 'train_loss': 0.019361726505060992, 'epoch': 2.0})

## モデルの保存

In [ ]:
import os

# 1. 保存パスの設定（struct-eval-comp プロジェクト用）
adapter_name = "struct-eval-comp-dpo-v10"
drive_save_path = f"/content/drive/MyDrive/LLM2026/main_competition/DPO/{adapter_name}"
hf_model_id = "satoyutaka/LLM2025_main_0_DPO10" # 自分のHFユーザー名に合わせてね

# 2. Google ドライブへ保存
print(f"💾 Googleドライブに保存中: {drive_save_path}")
os.makedirs(drive_save_path, exist_ok=True)
model.save_pretrained(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

# 3. Hugging Face へアップロード
print(f"🚀 Hugging Faceにアップロード中: {hf_model_id}")
try:
    model.push_to_hub(hf_model_id)
    tokenizer.push_to_hub(hf_model_id)
    print("✅ HFへのアップロードも完了したよ！")
except Exception as e:
    print(f"⚠️ HFアップロードでエラーが出たけど、ドライブには保存されてるよ: {e}")

print("\n✨ すべての保存工程が完了しました！お疲れ様、ゆーちゃ！")


💾 Googleドライブに保存中: /content/drive/MyDrive/LLM2026/main_competition/DPO/struct-eval-comp-dpo-v10
🚀 Hugging Faceにアップロード中: satoyutaka/LLM2025_main_0_DPO10
⚠️ HFアップロードでエラーが出たけど、ドライブには保存されてるよ: (Request ID: Root=1-6987be1d-75ab81793206e17845e6a58e;be09a801-a743-4ca2-90d2-5e56e8971238)

403 Forbidden: You don't have the rights to create a model under the namespace "satoyutaka".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.

✨ すべての保存工程が完了しました！お疲れ様、ゆーちゃ！


# 11回目

🚀 DPO学習：V11 最終一点突破（低学習率・高圧スタイルの融合）
このセルでは、V10までの試行錯誤をすべて凝縮した**「V11：ハイブリッド・スパルタ」**を実行します。 Qwenの強固なMarkdownバイアスを打破するため、あえて全層（All Modules）をターゲットに戻しつつ、学習率を極限まで下げることで、既存の構造化能力（0.77水準）を固定したまま形式だけを矯正します。

1. V11 の設計思想：なぜ「全層 ＋ 低Beta ＋ 低LR」なのか？

全層ターゲット: V10（MLP限定）ではMarkdown癖が抜けなかったことから、このバイアスはAttention層の出力生成ロジックにも深く根ざしていると判断しました。

低Beta (0.025): V10 (0.035) で変化がなかったため、さらに圧力を強めます。ただし、崩壊を招いた 0.01 までは下げず、知能を維持できるギリギリのラインを攻めます。

低学習率 (5e-7): 通常の半分以下の学習率に設定します。これにより、モデルの脳を急激に書き換えるのではなく、「今の賢さを保ったまま、バッククォートだけをソッと外す」という繊細な調整を可能にします。

2. 学習の期待値

V11の狙いは、V7/V9/V10で停滞している「バッククォート119件」という壁を物理的に破壊することです。 論理構造を司るAttention層を「微振動」させることで、構造を保ったまま、不要な装飾（Markdown）だけを削ぎ落とし、スコア 0.80 の大台への到達を目指します。

In [ ]:
# 1. ライブラリを最新に更新して、衝突を解消するよ！
!pip install --upgrade huggingface_hub unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.7/405.7 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 kB 16.8 MB/s eta 0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.13.2
    Uninstalling peft-0.13.2:
      Successfully uninstalled peft-0.13.2
  Attempting uninstall: unsloth_zoo
    Found existing installation: unsloth_zoo 2025.12.7
    Uninstalling unsloth_zoo-2025.12.7:
      Successfully uninstalled unsloth_zoo-2025.12.7
  Attempting uninstall: unsloth
    Found existing installation: unsloth 2025.12.7
    Uninstalling unsloth-2025.12.7:
      Successfully uninstalled unsloth-2025.12.7


## データのロード

In [ ]:
from datasets import load_dataset

# 1. 保存したjsonlファイルを読み込む
# 先ほどのスクリプトと同じディレクトリにあることを想定しているよ
data_files = {
    "train": "/content/drive/MyDrive/LLM2026/main_competition/DPO_data/struct_special_7_dpo_train.jsonl",
    "eval": "/content/drive/MyDrive/LLM2026/main_competition/DPO_data/struct_special_7_dpo_eval.jsonl"
}

# json形式でロード
dataset = load_dataset("json", data_files=data_files)

# 2. train と eval に切り分け
train_dataset = dataset["train"]
eval_dataset = dataset["eval"]

# 3. 中身の確認（デバッグ用：最初の1件を表示）
print("--- Train Dataset Sample ---")
print(train_dataset[0])
print(f"\nTrain size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")




--- Train Dataset Sample ---
{'prompt': '### Instruction:\n<|im_start|>system\nYou are a helpful assistant. Please format your response as follows:\nApproach: <step-by-step reasoning>\nOutput: <final answer><|im_end|>\n<|im_start|>user\nOutput api specification as XML.<|im_end|>\n<|im_start|>assistant\n\n\n### Response:\n', 'chosen': 'Approach:\n1. Task: Create api specification in XML\n2. Complexity: medium - 5-8 fields, 2-3 levels\n3. Format rules: well-formed tags, proper nesting, escaped special chars\n4. Structure: Organize data logically with appropriate nesting\n5. Populate fields with realistic example data based on the schema.\n\nOutput:\n<?xml version="1.0" encoding="UTF-8"?>\n<api_specification>\n<openapi>3.0.0</openapi>\n<info>\n<title>Multi-lateral demand-driven matrices API</title>\n<version>1.0.0</version>\n</info>\n<paths>\n</speech>\n<get>\n<summary>According institution everybody bad listen nearly.</summary>\n<parameters>\n<name>param0</name>\n<in>query</in>\n<type>st

## モデルのロードおよび学習

In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer
import torch
from peft import set_peft_model_state_dict
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from trl import DPOConfig, DPOTrainer

# --- 1. 事前パッチ ---
PatchDPOTrainer()

# --- 2. モデルロード ---
base_model_id = "Qwen/Qwen3-4B-Instruct-2507"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_id,
    max_seq_length = 2048,
    load_in_4bit = True,
    trust_remote_code = True,
)

# # --- 3. LoRA層の追加 ---
# # Rank(r)を維持しつつ、学習効率を最大化するよ。
# model = FastLanguageModel.get_peft_model(
#     model,
#     r = 16,
#     target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
#     lora_alpha = 16,
#     lora_dropout = 0,
#     bias = "none",
#     use_gradient_checkpointing = "unsloth",
#     random_state = 3407,
# )
# --- V11: MLP限定・スタイル一点突破設定 ---
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["gate_proj", "up_proj", "down_proj"], # MLPに集中
    lora_alpha = 32, # 影響力を高める
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# Trainer設定：Betaを0.02まで攻める
# args.beta = 0.02
# --- 4. 0.77の重みを注入 (外科手術) ---
# 前回の最高傑作(DPO3/v6)の重みをロードして、ここをスタート地点にするよ！
print("🔄 前回の最高傑作（0.77）の知能を注入中...")
adapter_weights = hf_hub_download(repo_id="satoyutaka/LLM2025_main_0_DPO7", filename="adapter_model.safetensors")
state_dict = load_file(adapter_weights)
set_peft_model_state_dict(model, state_dict)

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,
    args = DPOConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        warmup_ratio = 0.1,
        num_train_epochs = 1,            # 知能を壊さないよう、短く鋭く1エポック
        learning_rate = 5e-7,            # 極限まで低くし、既存の知能を「固定」
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs_V11_surgical_strike",
        beta = 0.025,                    # V10より強く、V8より優しく。
        report_to = "none",
        remove_unused_columns = False,
        max_length = 1024,
        max_prompt_length = 512,
    ),
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    tokenizer = tokenizer,
)

print("🚀 0.80の壁を壊しにいこう！学習開始！")
dpo_trainer.train()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.1.4: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Not an error, but Unsloth cannot patch Attention layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch O projection layer with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.1.4 patched 36 layers with 0 QKV layers, 0 O layers and 36 MLP layers.


🔄 前回の最高傑作（0.77）の知能を注入中...
🚀 0.80の壁を壊しにいこう！学習開始！


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 225 | Num Epochs = 1 | Total steps = 15
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 21,233,664 of 4,043,701,760 (0.53% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
1,0.110300,2.037352,-0.332268,1.000000,2.369620,-349.894470,-310.842957,-3.661474,-3.781754,0,0,0
2,0.112700,1.883054,-0.399859,1.000000,2.282913,-359.825439,-265.425842,-3.603421,-3.958147,No Log,No Log,No Log
3,0.094000,2.185771,-0.239857,1.000000,2.425628,-385.468384,-214.747314,-3.520974,-3.987669,No Log,No Log,No Log
4,0.085100,2.159085,-0.339725,1.000000,2.498811,-356.399536,-134.605560,-3.549289,-4.091456,No Log,No Log,No Log
5,0.098900,2.020571,-0.298743,1.000000,2.319314,-372.408325,-157.708435,-3.358441,-4.136825,No Log,No Log,No Log
6,0.093500,2.028545,-0.374421,1.000000,2.402967,-404.534088,-161.400452,-3.450483,-4.047989,No Log,No Log,No Log
7,0.075700,2.229712,-0.396953,1.000000,2.626665,-375.072083,-118.625717,-3.624194,-3.967557,No Log,No Log,No Log
8,0.088900,2.036707,-0.431305,1.000000,2.468012,-362.221161,-136.485184,-3.683374,-3.803466,No Log,No Log,No Log
9,0.080600,2.204898,-0.365289,1.000000,2.570187,-384.763062,-117.559677,-3.778755,-4.030806,No Log,No Log,No Log
10,0.079500,2.184732,-0.384579,1.000000,2.569312,-386.973175,-119.954155,-3.556352,-3.933049,No Log,No Log,No Log


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
1,0.110300,2.037352,-0.332268,1.000000,2.369620,-349.894470,-310.842957,-3.661474,-3.781754,0,0,0
2,0.112700,1.883054,-0.399859,1.000000,2.282913,-359.825439,-265.425842,-3.603421,-3.958147,No Log,No Log,No Log
3,0.094000,2.185771,-0.239857,1.000000,2.425628,-385.468384,-214.747314,-3.520974,-3.987669,No Log,No Log,No Log
4,0.085100,2.159085,-0.339725,1.000000,2.498811,-356.399536,-134.605560,-3.549289,-4.091456,No Log,No Log,No Log
5,0.098900,2.020571,-0.298743,1.000000,2.319314,-372.408325,-157.708435,-3.358441,-4.136825,No Log,No Log,No Log
6,0.093500,2.028545,-0.374421,1.000000,2.402967,-404.534088,-161.400452,-3.450483,-4.047989,No Log,No Log,No Log
7,0.075700,2.229712,-0.396953,1.000000,2.626665,-375.072083,-118.625717,-3.624194,-3.967557,No Log,No Log,No Log
8,0.088900,2.036707,-0.431305,1.000000,2.468012,-362.221161,-136.485184,-3.683374,-3.803466,No Log,No Log,No Log
9,0.080600,2.204898,-0.365289,1.000000,2.570187,-384.763062,-117.559677,-3.778755,-4.030806,No Log,No Log,No Log
10,0.079500,2.184732,-0.384579,1.000000,2.569312,-386.973175,-119.954155,-3.556352,-3.933049,No Log,No Log,No Log


TrainOutput(global_step=15, training_loss=0.09078663537899653, metrics={'train_runtime': 755.9393, 'train_samples_per_second': 0.298, 'train_steps_per_second': 0.02, 'total_flos': 0.0, 'train_loss': 0.09078663537899653, 'epoch': 1.0})

## モデルの保存

In [ ]:
import os

# 1. 保存パスの設定（struct-eval-comp プロジェクト用）
adapter_name = "struct-eval-comp-dpo-v11"
drive_save_path = f"/content/drive/MyDrive/LLM2026/main_competition/DPO/{adapter_name}"
hf_model_id = "satoyutaka/LLM2025_main_0_DPO11" # 自分のHFユーザー名に合わせてね

# 2. Google ドライブへ保存
print(f"💾 Googleドライブに保存中: {drive_save_path}")
os.makedirs(drive_save_path, exist_ok=True)
model.save_pretrained(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

# 3. Hugging Face へアップロード
print(f"🚀 Hugging Faceにアップロード中: {hf_model_id}")
try:
    model.push_to_hub(hf_model_id)
    tokenizer.push_to_hub(hf_model_id)
    print("✅ HFへのアップロードも完了したよ！")
except Exception as e:
    print(f"⚠️ HFアップロードでエラーが出たけど、ドライブには保存されてるよ: {e}")

print("\n✨ すべての保存工程が完了しました！お疲れ様、ゆーちゃ！")


💾 Googleドライブに保存中: /content/drive/MyDrive/LLM2026/main_competition/DPO/struct-eval-comp-dpo-v11
🚀 Hugging Faceにアップロード中: satoyutaka/LLM2025_main_0_DPO11


README.md: 0.00B [00:00, ?B/s]

⚠️ HFアップロードでエラーが出たけど、ドライブには保存されてるよ: (Request ID: Root=1-6987c79a-077731cb7d57ff167a665576;fcb7dbe7-7c41-440e-b96a-df2a42b77420)

403 Forbidden: Forbidden: you must use a write token to upload to a repository..
Cannot access content at: https://huggingface.co/api/models/satoyutaka/LLM2025_main_0_DPO11/preupload/main.
Make sure your token has the correct permissions.

✨ すべての保存工程が完了しました！お疲れ様、ゆーちゃ！
